In [1]:
from pathlib import Path
import json
import sys
import re

from dataclasses import dataclass
from collections import Counter, defaultdict
import copy

from joblib import Parallel, delayed
import os
import time

import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance


project_dir = Path.cwd()

raw_dir = project_dir / "data" / "raw"
output_dir = project_dir / "outputs" / "odds_40"

results_path = raw_dir / "results.csv"
rankings_path = raw_dir / "fifa_consolidado.csv"
schedule_path = raw_dir / "world_cup_2026_schedule.csv"
odds_json_path = raw_dir / "odds_fase_grupos_copa.json"
odds_csv_path = raw_dir / "odds.csv"

output_dir.mkdir(parents=True, exist_ok=True)

print(project_dir)
print(raw_dir.exists())

@dataclass
class TournamentSimulationRounds:
    champion: str
    runner_up: str
    third_place: str
    fourth_place: str
    finalists: list[str]
    semifinalists: list[str]
    quarterfinalists: list[str]
    round_of_16_teams: list[str]
    round_of_32_teams: list[str]
    group_advancers: list[str]
    playoff_qualifiers: list[str]

d:\windows\Documents\extras\previsao_copa_26
True


In [2]:
TEAM_NAME_ALIASES = {
    # América do Norte / Caribe
    "USA": "United States",
    "United States of America": "United States",
    "US Virgin Islands": "United States Virgin Islands",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St Lucia": "Saint Lucia",
    "St. Lucia": "Saint Lucia",
    "St Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St. Vincent / Grenadines": "Saint Vincent and the Grenadines",
    "Curacao": "Curacao",
    "Curaçao": "Curacao",

    # África
    "Cape Verde": "Cabo Verde",
    "Cape Verde Islands": "Cabo Verde",
    "Cabo Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
    "Democratic Republic of Congo": "Congo DR",
    "Congo DR": "Congo DR",
    "Ivory Coast": "Ivory Coast",
    "Côte d'Ivoire": "Ivory Coast",
    "The Gambia": "Gambia",
    "São Tomé and Príncipe": "Sao Tome e Principe",
    "São Tomé e Príncipe": "Sao Tome e Principe",
    "Sao Tome e Principe": "Sao Tome e Principe",

    # Ásia
    "China PR": "China",
    "Hong Kong, China": "Hong Kong",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "IR Iran": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Brunei Darussalam": "Brunei",

    # Europa
    "Czech Republic": "Czechia",
    "Türkiye": "Turkey",
    "Republic of Ireland": "Ireland",
    "FYR Macedonia": "North Macedonia",
    "Serbia and Montenegro" : "Serbia",

    # Oceania
    "Aotearoa New Zealand": "New Zealand",

    # Casos com marcador "(unranked)"
    "American Samoa (unranked)": "American Samoa",
    "Eritrea (unranked)": "Eritrea",
    "Samoa (unranked)": "Samoa",
    "Tonga (unranked)": "Tonga",
}

In [3]:
def normalize_team_name_extended(team_name):
    if pd.isna(team_name):
        return team_name

    team_name = str(team_name).strip()

    # Remove notas de Wikipedia: [nb 3], [1], etc.
    team_name = re.sub(r"\[.*?\]", "", team_name).strip()

    # Remove marcadores comuns
    team_name = team_name.replace("(H)", "").strip()
    team_name = team_name.replace("(unranked)", "").strip()

    # Ajusta espaços duplicados
    team_name = re.sub(r"\s+", " ", team_name).strip()

    # Aplica alias explícito
    team_name = TEAM_NAME_ALIASES.get(team_name, team_name)

    return team_name    

In [4]:
TEAM_NAME_MAP = {
    "USA": "United States",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Ivory Coast",
    "Czech Republic": "Czechia",
    "Türkiye": "Turkey",
}


def clean_team_name(team_name):
    team_name = str(team_name).strip()
    return TEAM_NAME_MAP.get(team_name, team_name)


def extract_h2h_prices(event):
    home_team = clean_team_name(event["home_team"])
    away_team = clean_team_name(event["away_team"])

    home_prices = []
    draw_prices = []
    away_prices = []

    for bookmaker in event.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market.get("key") != "h2h":
                continue

            prices_by_name = {
                clean_team_name(outcome["name"]): outcome["price"]
                for outcome in market.get("outcomes", [])
            }

            if home_team in prices_by_name:
                home_prices.append(float(prices_by_name[home_team]))

            if away_team in prices_by_name:
                away_prices.append(float(prices_by_name[away_team]))

            if "Draw" in prices_by_name:
                draw_prices.append(float(prices_by_name["Draw"]))

    if not home_prices or not draw_prices or not away_prices:
        return None

    return {
        "event_date": pd.to_datetime(event["commence_time"]).date(),
        "event_home_team": home_team,
        "event_away_team": away_team,
        "event_home_win_odds": float(np.median(home_prices)),
        "event_draw_odds": float(np.median(draw_prices)),
        "event_away_win_odds": float(np.median(away_prices)),
        "n_bookmakers": len(home_prices),
    }


with odds_json_path.open("r", encoding="utf-8") as file:
    odds_events = json.load(file)

odds_rows = []

for event in odds_events:
    extracted_prices = extract_h2h_prices(event)

    if extracted_prices is not None:
        odds_rows.append(extracted_prices)

odds_data = pd.DataFrame(odds_rows)

print(odds_data.shape)
odds_data.head()

(72, 7)


,event_date,event_home_team,event_away_team,event_home_win_odds,event_draw_odds,event_away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-12,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-13,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [5]:
schedule_data = pd.read_csv(schedule_path)

schedule_data["home_team"] = schedule_data["home_team"].map(clean_team_name)
schedule_data["away_team"] = schedule_data["away_team"].map(clean_team_name)

aligned_rows = []

for _, odds_row in odds_data.iterrows():
    event_home_team = odds_row["event_home_team"]
    event_away_team = odds_row["event_away_team"]

    same_order = schedule_data[
        (schedule_data["home_team"] == event_home_team)
        & (schedule_data["away_team"] == event_away_team)
    ]

    reverse_order = schedule_data[
        (schedule_data["home_team"] == event_away_team)
        & (schedule_data["away_team"] == event_home_team)
    ]

    if not same_order.empty:
        schedule_row = same_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_home_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_away_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    elif not reverse_order.empty:
        schedule_row = reverse_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_away_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_home_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    else:
        print("Não encontrei na tabela:", event_home_team, "x", event_away_team)

odds_aligned_data = pd.DataFrame(aligned_rows)

odds_aligned_data.to_csv(odds_csv_path, index=False)

print(odds_aligned_data.shape)
odds_aligned_data.head()

(72, 7)


,match_date,home_team,away_team,home_win_odds,draw_odds,away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-11,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-12,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [6]:
import poison_26 as path_a

In [7]:
historical_matches_raw = pd.read_csv(results_path)
rankings_raw = pd.read_csv(rankings_path)
schedule_raw = pd.read_csv(schedule_path)
odds_raw = pd.read_csv(odds_csv_path)

historical_matches_raw = historical_matches_raw.copy()

score_columns = ["home_score", "away_score"]

# Histórico de jogos
for column in ["home_team", "away_team"]:
    if column in historical_matches_raw.columns:
        historical_matches_raw[column] = historical_matches_raw[column].map(
            normalize_team_name_extended
        )

# Ranking FIFA
possible_ranking_team_columns = [
    "team",
    "country_full",
    "country",
    "name",
    "nation",
]

for column in possible_ranking_team_columns:
    if column in rankings_raw.columns:
        rankings_raw[column] = rankings_raw[column].map(
            normalize_team_name_extended
        )

# Tabela da Copa
for column in ["home_team", "away_team"]:
    if column in schedule_raw.columns:
        schedule_raw[column] = schedule_raw[column].map(
            normalize_team_name_extended
        )

# Odds, se estiver usando
if odds_csv_path.exists():
    odds_raw = pd.read_csv(odds_csv_path)

    for column in ["home_team", "away_team"]:
        if column in odds_raw.columns:
            odds_raw[column] = odds_raw[column].map(
                normalize_team_name_extended
            )

    odds_raw.to_csv(odds_csv_path, index=False)

historical_matches_raw["date"] = pd.to_datetime(
    historical_matches_raw["date"],
    errors="coerce",
)

historical_matches_raw["home_score"] = pd.to_numeric(
    historical_matches_raw["home_score"],
    errors="coerce",
)

historical_matches_raw["away_score"] = pd.to_numeric(
    historical_matches_raw["away_score"],
    errors="coerce",
)

missing_score_data = historical_matches_raw[
    historical_matches_raw[score_columns].isna().any(axis=1)
].copy()

print("Jogos sem placar:", missing_score_data.shape[0])

display(
    missing_score_data[
        ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]
    ].tail(20)
)

historical_matches_clean = historical_matches_raw.dropna(
    subset=["date", "home_score", "away_score"]
).copy()

historical_matches_clean = historical_matches_clean[
    historical_matches_clean["date"] < pd.Timestamp("2026-06-11")
].copy()

historical_matches_clean["home_score"] = historical_matches_clean["home_score"].astype(int)
historical_matches_clean["away_score"] = historical_matches_clean["away_score"].astype(int)

historical_matches = path_a.standardize_matches_columns(historical_matches_clean)
schedule_data = path_a.standardize_schedule_columns(schedule_raw)

ranking_lookup = path_a.RankingLookup(rankings_raw)
odds_lookup = path_a.OddsLookup(odds_raw)

print("Histórico bruto:", historical_matches_raw.shape)
print("Histórico limpo:", historical_matches.shape)
print("Tabela Copa:", schedule_data.shape)
print("Odds:", odds_raw.shape)

Jogos sem placar: 72


,date,home_team,away_team,home_score,away_score,tournament
49452,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup
49453,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup
49454,2026-06-25,United States,Turkey,NaN,NaN,FIFA World Cup
49455,2026-06-25,Paraguay,Australia,NaN,NaN,FIFA World Cup
49456,2026-06-25,Curacao,Ivory Coast,NaN,NaN,FIFA World Cup
49457,2026-06-25,Ecuador,Germany,NaN,NaN,FIFA World Cup
49458,2026-06-25,Japan,Sweden,NaN,NaN,FIFA World Cup
49459,2026-06-25,Tunisia,Netherlands,NaN,NaN,FIFA World Cup
49460,2026-06-26,Egypt,Iran,NaN,NaN,FIFA World Cup
49461,2026-06-26,New Zealand,Belgium,NaN,NaN,FIFA World Cup


Histórico bruto: (49472, 9)
Histórico limpo: (49400, 9)
Tabela Copa: (72, 8)
Odds: (72, 7)


In [8]:
market_features = [
    "market_home_prob",
    "market_draw_prob",
    "market_away_prob",
]

path_a.FEATURE_COLUMNS = [
    feature
    for feature in path_a.FEATURE_COLUMNS
    if feature not in market_features
]

odds_lookup_for_training = path_a.OddsLookup(None)

features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup_for_training,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(features_data.shape)
print(path_a.FEATURE_COLUMNS)

(49400, 23)
['elo_diff', 'attack_diff', 'defense_diff', 'form_diff', 'home_rank', 'away_rank', 'rank_diff', 'home_total_points', 'away_total_points', 'fifa_points_diff', 'neutral', 'is_non_neutral', 'tournament_weight', 'home_attack', 'away_attack', 'home_defense', 'away_defense', 'home_form', 'away_form', 'home_decay', 'away_decay', 'home_matches_played', 'away_matches_played']


In [9]:
features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(home_goals.shape)
print(away_goals.shape)

(49400,)
(49400,)


In [10]:
def normalize_team_name_safe(team):
    if hasattr(path_a, "normalize_team_name"):
        return path_a.normalize_team_name(team)

    return str(team).strip()

def extract_teams_from_rankings(rankings_raw):
    possible_team_columns = [
        "team",
        "country_full",
        "country",
        "name",
        "nation",
    ]

    for column in possible_team_columns:
        if column in rankings_raw.columns:
            return set(
                rankings_raw[column]
                .dropna()
                .map(normalize_team_name_safe)
                .unique()
            )

    return set()

def get_all_available_teams(
    initial_states,
    historical_matches,
    rankings_raw,
):
    teams = set()

    teams.update(initial_states.keys())

    if "home_team" in historical_matches.columns:
        teams.update(historical_matches["home_team"].dropna().map(normalize_team_name_safe))

    if "away_team" in historical_matches.columns:
        teams.update(historical_matches["away_team"].dropna().map(normalize_team_name_safe))

    teams.update(extract_teams_from_rankings(rankings_raw))

    teams = sorted(
        team
        for team in teams
        if isinstance(team, str)
        and team.strip() != ""
        and team.lower() != "nan"
    )

    return teams

all_teams_after_normalization = get_all_available_teams(
    initial_states=initial_states,
    historical_matches=historical_matches,
    rankings_raw=rankings_raw,
)

suspicious_terms = [
    "Czech",
    "Congo",
    "Cape",
    "Cabo",
    "Cura",
    "Korea",
    "Iran",
    "Ireland",
    "Macedonia",
    "Turkey",
    "Türkiye",
    "Saint",
    "St ",
    "Tome",
    "Príncipe",
    "Principe",
    "United States",
    "USA",
]

for term in suspicious_terms:
    matches = [
        team for team in all_teams_after_normalization
        if term.lower() in team.lower()
    ]

    if matches:
        print("\n", term)
        print(matches)

all_teams = get_all_available_teams(
    initial_states=initial_states,
    historical_matches=historical_matches,
    rankings_raw=rankings_raw,
)

print("Total de seleções encontradas:", len(all_teams))
print(all_teams[:30])


 Czech
['Czechia', 'Czechoslovakia']

 Congo
['Congo', 'Congo DR']

 Cabo
['Cabo Verde']

 Cura
['Curacao']

 Korea
['DPR Korea', 'North Korea', 'South Korea', 'United Koreans in Japan']

 Iran
['Iran']

 Ireland
['Ireland', 'Northern Ireland']

 Macedonia
['North Macedonia']

 Turkey
['Turkey']

 Saint
['Saint Barthélemy', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Martin', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines']

 St 
['East Turkestan', 'West Papua']

 Tome
['Sao Tome e Principe']

 Principe
['Sao Tome e Principe']

 United States
['United States', 'United States Virgin Islands']
Total de seleções encontradas: 341
['Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria', 'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Argentina', 'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Australia', 'Austria', 'Aymara', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barawa'

In [11]:
def calculate_poisson_goal_probabilities(goal_rate, max_goals=10):
    probabilities = np.zeros(max_goals + 1)
    probabilities[0] = np.exp(-goal_rate)

    for goals in range(1, max_goals + 1):
        probabilities[goals] = probabilities[goals - 1] * goal_rate / goals

    return probabilities / probabilities.sum()


def calculate_score_probability_matrix(home_lambda, away_lambda, max_goals=10):
    home_goal_probabilities = calculate_poisson_goal_probabilities(
        home_lambda,
        max_goals=max_goals,
    )

    away_goal_probabilities = calculate_poisson_goal_probabilities(
        away_lambda,
        max_goals=max_goals,
    )

    score_matrix = np.outer(
        home_goal_probabilities,
        away_goal_probabilities,
    )

    return score_matrix / score_matrix.sum()


def calculate_outcome_probabilities_from_score_matrix(score_matrix):
    home_win_probability = np.tril(score_matrix, k=-1).sum()
    draw_probability = np.trace(score_matrix)
    away_win_probability = np.triu(score_matrix, k=1).sum()

    total_probability = (
        home_win_probability
        + draw_probability
        + away_win_probability
    )

    return {
        "home_win": home_win_probability / total_probability,
        "draw": draw_probability / total_probability,
        "away_win": away_win_probability / total_probability,
    }


In [12]:
def predict_poisson_pure_match(
    home_team,
    away_team,
    initial_states,
    ranking_lookup,
    home_model,
    away_model,
    match_date="2026-06-10",
    neutral=1,
    tournament="FIFA World Cup",
    max_goals=10,
):
    market_probabilities = path_a.DEFAULT_MARKET_PROBABILITIES.copy()

    feature_row = path_a.build_feature_row(
        home_team=home_team,
        away_team=away_team,
        match_date=pd.Timestamp(match_date),
        neutral=neutral,
        tournament=tournament,
        states=initial_states,
        ranking_lookup=ranking_lookup,
        market_probabilities=market_probabilities,
    )

    home_lambda, away_lambda = path_a.predict_expected_goals(
        home_model,
        away_model,
        feature_row,
    )

    score_matrix = calculate_score_probability_matrix(
        home_lambda=home_lambda,
        away_lambda=away_lambda,
        max_goals=max_goals,
    )

    probabilities = calculate_outcome_probabilities_from_score_matrix(score_matrix)

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_lambda": home_lambda,
        "away_lambda": away_lambda,
        "home_win_probability": probabilities["home_win"],
        "draw_probability": probabilities["draw"],
        "away_win_probability": probabilities["away_win"],
        "expected_goal_diff_home": home_lambda - away_lambda,
    }

In [13]:
def compare_team_vs_italy_pure_poisson(
    team,
    baseline_team,
    initial_states,
    ranking_lookup,
    home_model,
    away_model,
    match_date="2026-06-11",
    max_goals=10,
):
    italy_home = predict_poisson_pure_match(
        home_team=baseline_team,
        away_team=team,
        initial_states=initial_states,
        ranking_lookup=ranking_lookup,
        home_model=home_model,
        away_model=away_model,
        match_date=match_date,
        neutral=1,
        tournament="FIFA World Cup",
        max_goals=max_goals,
    )

    italy_away = predict_poisson_pure_match(
        home_team=team,
        away_team=baseline_team,
        initial_states=initial_states,
        ranking_lookup=ranking_lookup,
        home_model=home_model,
        away_model=away_model,
        match_date=match_date,
        neutral=1,
        tournament="FIFA World Cup",
        max_goals=max_goals,
    )

    team_points_when_italy_home = (
        3 * italy_home["away_win_probability"]
        + italy_home["draw_probability"]
    )

    italy_points_when_italy_home = (
        3 * italy_home["home_win_probability"]
        + italy_home["draw_probability"]
    )

    team_points_when_italy_away = (
        3 * italy_away["home_win_probability"]
        + italy_away["draw_probability"]
    )

    italy_points_when_italy_away = (
        3 * italy_away["away_win_probability"]
        + italy_away["draw_probability"]
    )

    team_win_when_italy_home = italy_home["away_win_probability"]
    team_draw_when_italy_home = italy_home["draw_probability"]
    team_loss_when_italy_home = italy_home["home_win_probability"]

    team_win_when_italy_away = italy_away["home_win_probability"]
    team_draw_when_italy_away = italy_away["draw_probability"]
    team_loss_when_italy_away = italy_away["away_win_probability"]

    avg_team_points = (
        team_points_when_italy_home
        + team_points_when_italy_away
    ) / 2

    avg_italy_points = (
        italy_points_when_italy_home
        + italy_points_when_italy_away
    ) / 2

    avg_team_win_probability = (
        team_win_when_italy_home
        + team_win_when_italy_away
    ) / 2

    avg_draw_probability = (
        team_draw_when_italy_home
        + team_draw_when_italy_away
    ) / 2

    avg_team_loss_probability = (
        team_loss_when_italy_home
        + team_loss_when_italy_away
    ) / 2

    avg_goal_diff_team = (
        -italy_home["expected_goal_diff_home"]
        + italy_away["expected_goal_diff_home"]
    ) / 2

    avg_team_expected_goals = (
        italy_home["away_lambda"]
        + italy_away["home_lambda"]
    ) / 2

    avg_italy_expected_goals = (
        italy_home["home_lambda"]
        + italy_away["away_lambda"]
    ) / 2

    return {
        "team": team,
        "baseline_team": baseline_team,

        "avg_expected_points_vs_italy": avg_team_points,
        "avg_italy_expected_points": avg_italy_points,
        "avg_win_probability_vs_italy": avg_team_win_probability,
        "avg_draw_probability_vs_italy": avg_draw_probability,
        "avg_loss_probability_vs_italy": avg_team_loss_probability,
        "avg_expected_goal_diff_vs_italy": avg_goal_diff_team,
        "avg_team_expected_goals": avg_team_expected_goals,
        "avg_italy_expected_goals": avg_italy_expected_goals,

        "team_points_when_italy_home": team_points_when_italy_home,
        "team_win_probability_when_italy_home": team_win_when_italy_home,
        "draw_probability_when_italy_home": team_draw_when_italy_home,
        "team_loss_probability_when_italy_home": team_loss_when_italy_home,
        "team_expected_goals_when_italy_home": italy_home["away_lambda"],
        "italy_expected_goals_when_italy_home": italy_home["home_lambda"],

        "team_points_when_italy_away": team_points_when_italy_away,
        "team_win_probability_when_italy_away": team_win_when_italy_away,
        "draw_probability_when_italy_away": team_draw_when_italy_away,
        "team_loss_probability_when_italy_away": team_loss_when_italy_away,
        "team_expected_goals_when_italy_away": italy_away["home_lambda"],
        "italy_expected_goals_when_italy_away": italy_away["away_lambda"],
    }


baseline_team = "Italy"

rows = []
errors = []

for team in all_teams:
    if team == baseline_team:
        continue

    try:
        row = compare_team_vs_italy_pure_poisson(
            team=team,
            baseline_team=baseline_team,
            initial_states=initial_states,
            ranking_lookup=ranking_lookup,
            home_model=home_model,
            away_model=away_model,
            match_date="2026-06-11",
            max_goals=10,
        )

        rows.append(row)

    except Exception as error:
        errors.append(
            {
                "team": team,
                "error": str(error),
            }
        )

ranking_vs_italy_all = pd.DataFrame(rows)

ranking_vs_italy_all = ranking_vs_italy_all.sort_values(
    [
        "avg_expected_points_vs_italy",
        "avg_expected_goal_diff_vs_italy",
        "avg_win_probability_vs_italy",
    ],
    ascending=False,
).reset_index(drop=True)

ranking_vs_italy_all["rank_vs_italy"] = ranking_vs_italy_all.index + 1

ranking_vs_italy_all = ranking_vs_italy_all[
    ["rank_vs_italy"]
    + [
        column
        for column in ranking_vs_italy_all.columns
        if column != "rank_vs_italy"
    ]
]

errors_vs_italy = pd.DataFrame(errors)

print("Seleções avaliadas:", ranking_vs_italy_all.shape[0])
print("Seleções com erro:", errors_vs_italy.shape[0])

display(ranking_vs_italy_all.head(30))
display(errors_vs_italy.head(30))

Seleções avaliadas: 340
Seleções com erro: 0


,rank_vs_italy,team,baseline_team,avg_expected_points_vs_italy,avg_italy_expected_points,avg_win_probability_vs_italy,avg_draw_probability_vs_italy,avg_loss_probability_vs_italy,avg_expected_goal_diff_vs_italy,avg_team_expected_goals,...,draw_probability_when_italy_home,team_loss_probability_when_italy_home,team_expected_goals_when_italy_home,italy_expected_goals_when_italy_home,team_points_when_italy_away,team_win_probability_when_italy_away,draw_probability_when_italy_away,team_loss_probability_when_italy_away,team_expected_goals_when_italy_away,italy_expected_goals_when_italy_away
0,1,Spain,Italy,2.313713,0.515231,0.714219,0.171056,0.114725,1.528707,2.362249,...,0.167686,0.110368,2.392370,0.822282,2.293901,0.706492,0.174425,0.119083,2.332128,0.844801
1,2,Argentina,Italy,2.061351,0.710637,0.611113,0.228013,0.160875,0.997304,1.782681,...,0.224772,0.156389,1.809323,0.778963,2.041412,0.603386,0.231254,0.165360,1.756039,0.791792
2,3,France,Italy,1.970367,0.801723,0.580819,0.227910,0.191271,0.880235,1.816851,...,0.225261,0.184405,1.839952,0.919371,1.944470,0.571303,0.230559,0.198137,1.793751,0.953862
3,4,Brazil,Italy,1.950527,0.812025,0.571027,0.237448,0.191526,0.834226,1.715882,...,0.233689,0.179794,1.746280,0.848839,1.907812,0.555535,0.241207,0.203258,1.685484,0.914472
4,5,England,Italy,1.854824,0.889208,0.532952,0.255968,0.211080,0.679537,1.548817,...,0.253615,0.205789,1.569734,0.860271,1.834244,0.525307,0.258322,0.216371,1.527900,0.878289
5,6,Colombia,Italy,1.836971,0.923625,0.532522,0.239404,0.228074,0.677639,1.709220,...,0.239689,0.230681,1.706316,1.040712,1.845364,0.535415,0.239120,0.225466,1.712125,1.022450
6,7,Germany,Italy,1.827972,0.929491,0.528478,0.242537,0.228985,0.660738,1.674822,...,0.240217,0.216376,1.699205,0.976843,1.785505,0.513550,0.244857,0.241594,1.650438,1.051326
7,8,Netherlands,Italy,1.802856,0.950169,0.518627,0.246975,0.234398,0.620661,1.630114,...,0.245640,0.228499,1.644618,0.994381,1.782489,0.511393,0.248310,0.240297,1.615610,1.024525
8,9,Portugal,Italy,1.769532,0.989804,0.509622,0.240665,0.249713,0.581424,1.693087,...,0.240217,0.248967,1.698228,1.111851,1.766400,0.508429,0.241112,0.250459,1.687947,1.111476
9,10,Belgium,Italy,1.759805,0.995418,0.505009,0.244778,0.250213,0.563160,1.647806,...,0.243386,0.240707,1.666022,1.057383,1.728501,0.494110,0.246170,0.259720,1.629590,1.111908


""


In [14]:
def get_team_state_summary(team, initial_states, ranking_lookup, match_date="2026-06-11"):
    state = initial_states[team]
    fifa_rank, fifa_points = ranking_lookup.get(
        team,
        pd.Timestamp(match_date),
    )

    return {
        "team": team,
        "elo": state.elo,
        "attack": state.attack,
        "defense": state.defense,
        "form": state.form,
        "matches_played": state.matches_played,
        "fifa_rank": fifa_rank,
        "fifa_points": fifa_points,
    }


spain_state = get_team_state_summary(
    team="Spain",
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
)

italy_state = get_team_state_summary(
    team="Italy",
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
)

state_comparison = pd.DataFrame([spain_state, italy_state])

display(state_comparison)

,team,elo,attack,defense,form,matches_played,fifa_rank,fifa_points
0,Spain,2246.827800,3.247003,0.633045,0.780562,783,2.0,1874.71
1,Italy,1966.599077,2.117306,0.921178,0.690756,893,12.0,1704.73


In [15]:
def split_simulations(total_simulations, n_chunks):
    base_size = total_simulations // n_chunks
    remainder = total_simulations % n_chunks

    chunk_sizes = []

    for chunk_index in range(n_chunks):
        chunk_size = base_size + (1 if chunk_index < remainder else 0)

        if chunk_size > 0:
            chunk_sizes.append(chunk_size)

    return chunk_sizes

In [16]:
def extract_poisson_weights(model, model_name):
    scaler = model.named_steps["scaler"]
    poisson = model.named_steps["poisson"]

    standardized_coefficients = pd.Series(
        poisson.coef_,
        index=path_a.FEATURE_COLUMNS,
        name="standardized_coefficient",
    )

    raw_scale_coefficients = standardized_coefficients / scaler.scale_

    weights_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "standardized_coefficient": standardized_coefficients.values,
            "raw_scale_coefficient": raw_scale_coefficients.values,
            "abs_standardized_coefficient": standardized_coefficients.abs().values,
        }
    )

    weights_data["model"] = model_name

    return weights_data.sort_values(
        "abs_standardized_coefficient",
        ascending=False,
    ).reset_index(drop=True)


home_weights = extract_poisson_weights(home_model, "home_goals")
away_weights = extract_poisson_weights(away_model, "away_goals")

display(home_weights.head(30))
display(away_weights.head(30))

,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,0.355819,0.001519,0.355819,home_goals
1,away_defense,0.112705,0.151912,0.112705,home_goals
2,away_matches_played,-0.104957,-0.000470,0.104957,home_goals
3,form_diff,-0.104567,-0.499397,0.104567,home_goals
4,home_attack,0.092997,0.154063,0.092997,home_goals
5,defense_diff,0.083969,0.099869,0.083969,home_goals
6,away_form,0.078900,0.494345,0.078900,home_goals
7,attack_diff,0.064220,0.090172,0.064220,home_goals
8,home_form,-0.058116,-0.363102,0.058116,home_goals
9,away_decay,-0.044172,-0.276153,0.044172,home_goals


,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,-0.359674,-0.001536,0.359674,away_goals
1,home_defense,0.128500,0.183187,0.128500,away_goals
2,away_attack,0.117326,0.196387,0.117326,away_goals
3,form_diff,0.097827,0.467208,0.097827,away_goals
4,defense_diff,-0.089568,-0.106527,0.089568,away_goals
5,away_form,-0.077091,-0.483010,0.077091,away_goals
6,home_matches_played,-0.055997,-0.000241,0.055997,away_goals
7,home_attack,0.055114,0.091305,0.055114,away_goals
8,attack_diff,-0.051705,-0.072600,0.051705,away_goals
9,home_form,0.051103,0.319284,0.051103,away_goals


In [17]:
def calculate_permutation_importance(model, features_data, target, model_name):
    result = permutation_importance(
        model,
        features_data[path_a.FEATURE_COLUMNS],
        target,
        n_repeats=10,
        random_state=42,
        scoring="neg_mean_absolute_error",
    )

    importance_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
            "model": model_name,
        }
    )

    return importance_data.sort_values(
        "importance_mean",
        ascending=False,
    ).reset_index(drop=True)


home_importance = calculate_permutation_importance(
    home_model,
    features_data,
    home_goals,
    "home_goals",
)

away_importance = calculate_permutation_importance(
    away_model,
    features_data,
    away_goals,
    "away_goals",
)

display(home_importance.head(30))
display(away_importance.head(30))

,feature,importance_mean,importance_std,model
0,elo_diff,0.201218,0.001297,home_goals
1,form_diff,0.039493,0.001065,home_goals
2,away_defense,0.024965,0.000771,home_goals
3,away_form,0.022890,0.000801,home_goals
4,home_attack,0.013188,0.000519,home_goals
5,away_matches_played,0.010738,0.000938,home_goals
6,home_form,0.009877,0.000500,home_goals
7,defense_diff,0.009498,0.000494,home_goals
8,fifa_points_diff,0.004633,0.000229,home_goals
9,attack_diff,0.002875,0.000548,home_goals


,feature,importance_mean,importance_std,model
0,elo_diff,0.116540,0.002086,away_goals
1,form_diff,0.022810,0.000524,away_goals
2,home_defense,0.016522,0.000638,away_goals
3,away_form,0.011200,0.000377,away_goals
4,away_attack,0.009800,0.000624,away_goals
5,home_form,0.007121,0.000326,away_goals
6,home_attack,0.005620,0.000388,away_goals
7,defense_diff,0.004391,0.000438,away_goals
8,fifa_points_diff,0.002605,0.000132,away_goals
9,neutral,0.002340,0.000354,away_goals


In [18]:
MARKET_WEIGHT = 0.4
MAX_GOALS = 10


def calculate_poisson_probabilities(goal_rate, max_goals=MAX_GOALS):
    probabilities = np.zeros(max_goals + 1)
    probabilities[0] = np.exp(-goal_rate)

    for goals in range(1, max_goals + 1):
        probabilities[goals] = probabilities[goals - 1] * goal_rate / goals

    probabilities = probabilities / probabilities.sum()

    return probabilities


def calculate_poisson_outcome_probabilities(
    home_lambda,
    away_lambda,
    max_goals=MAX_GOALS,
):
    home_probabilities = calculate_poisson_probabilities(home_lambda, max_goals)
    away_probabilities = calculate_poisson_probabilities(away_lambda, max_goals)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    home_win_probability = np.tril(score_matrix, k=-1).sum()
    draw_probability = np.trace(score_matrix)
    away_win_probability = np.triu(score_matrix, k=1).sum()

    total_probability = (
        home_win_probability
        + draw_probability
        + away_win_probability
    )

    return {
        "home_win": home_win_probability / total_probability,
        "draw": draw_probability / total_probability,
        "away_win": away_win_probability / total_probability,
    }


def normalize_probabilities(probabilities):
    total_probability = sum(probabilities.values())

    if total_probability <= 0:
        raise ValueError("A soma das probabilidades precisa ser positiva.")

    return {
        key: value / total_probability
        for key, value in probabilities.items()
    }


def market_probabilities_to_outcome_probabilities(market_probabilities):
    return normalize_probabilities(
        {
            "home_win": market_probabilities["market_home_prob"],
            "draw": market_probabilities["market_draw_prob"],
            "away_win": market_probabilities["market_away_prob"],
        }
    )


def has_real_market_probabilities(market_probabilities):
    default_probabilities = path_a.DEFAULT_MARKET_PROBABILITIES

    return any(
        abs(market_probabilities[key] - default_probabilities[key]) > 1e-10
        for key in default_probabilities
    )


def blend_outcome_probabilities(
    model_probabilities,
    market_probabilities,
    market_weight=MARKET_WEIGHT,
):
    if not has_real_market_probabilities(market_probabilities):
        return normalize_probabilities(model_probabilities)

    market_outcome_probabilities = market_probabilities_to_outcome_probabilities(
        market_probabilities
    )

    blended_probabilities = {
        outcome: (
            (1.0 - market_weight) * model_probabilities[outcome]
            + market_weight * market_outcome_probabilities[outcome]
        )
        for outcome in ["home_win", "draw", "away_win"]
    }

    return normalize_probabilities(blended_probabilities)

In [19]:
def get_score_outcome(home_goals, away_goals):
    if home_goals > away_goals:
        return "home_win"

    if home_goals == away_goals:
        return "draw"

    return "away_win"


def sample_score_from_blended_probabilities(
    home_lambda,
    away_lambda,
    target_probabilities,
    random_generator,
    max_goals=MAX_GOALS,
):
    home_probabilities = calculate_poisson_probabilities(home_lambda, max_goals)
    away_probabilities = calculate_poisson_probabilities(away_lambda, max_goals)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    model_outcome_probabilities = {
        "home_win": 0.0,
        "draw": 0.0,
        "away_win": 0.0,
    }

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            outcome = get_score_outcome(home_goals, away_goals)
            model_outcome_probabilities[outcome] += score_matrix[
                home_goals,
                away_goals,
            ]

    adjusted_score_matrix = score_matrix.copy()

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            outcome = get_score_outcome(home_goals, away_goals)
            model_probability = model_outcome_probabilities[outcome]

            if model_probability > 0:
                adjustment_factor = (
                    target_probabilities[outcome] / model_probability
                )
                adjusted_score_matrix[home_goals, away_goals] *= adjustment_factor

    adjusted_score_matrix = adjusted_score_matrix / adjusted_score_matrix.sum()

    flat_probabilities = adjusted_score_matrix.ravel()
    selected_index = random_generator.choice(
        len(flat_probabilities),
        p=flat_probabilities,
    )

    home_goals, away_goals = np.unravel_index(
        selected_index,
        adjusted_score_matrix.shape,
    )

    return int(home_goals), int(away_goals)

In [20]:
def simulate_match_with_market_blend(
    home_team,
    away_team,
    match_date,
    neutral,
    tournament,
    stage,
    group,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
    knockout=False,
):
    market_probabilities = odds_lookup.get(
        match_date,
        home_team,
        away_team,
    )

    feature_row = path_a.build_feature_row(
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        neutral=neutral,
        tournament=tournament,
        states=states,
        ranking_lookup=ranking_lookup,
        market_probabilities=market_probabilities,
    )

    home_lambda, away_lambda = path_a.predict_expected_goals(
        home_model,
        away_model,
        feature_row,
    )

    model_probabilities = calculate_poisson_outcome_probabilities(
        home_lambda,
        away_lambda,
    )

    target_probabilities = blend_outcome_probabilities(
        model_probabilities=model_probabilities,
        market_probabilities=market_probabilities,
        market_weight=MARKET_WEIGHT,
    )

    home_goals, away_goals = sample_score_from_blended_probabilities(
        home_lambda=home_lambda,
        away_lambda=away_lambda,
        target_probabilities=target_probabilities,
        random_generator=random_generator,
    )

    winner = path_a.infer_winner_from_score(
        home_team,
        away_team,
        home_goals,
        away_goals,
    )

    home_result_override = None

    if knockout and winner is None:
        home_advancement_probability = (
            target_probabilities["home_win"]
            / (
                target_probabilities["home_win"]
                + target_probabilities["away_win"]
            )
        )

        home_advancement_probability = float(
            np.clip(home_advancement_probability, 0.35, 0.65)
        )

        if random_generator.random() < home_advancement_probability:
            winner = home_team
            home_result_override = 0.55
        else:
            winner = away_team
            home_result_override = 0.45

    path_a.update_team_states(
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        match_date=match_date,
        tournament=tournament,
        states=states,
        home_result_override=home_result_override,
    )

    return path_a.MatchSimulation(
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        winner=winner,
        stage=stage,
        group=group,
    )


path_a.simulate_match = simulate_match_with_market_blend

In [21]:
ROUND_OF_32_RULES = {
    "M73": ("2A", "2B"),
    "M74": ("1E", "3"),
    "M75": ("1F", "2C"),
    "M76": ("1C", "2F"),
    "M77": ("1I", "3"),
    "M78": ("2E", "2I"),
    "M79": ("1A", "3"),
    "M80": ("1L", "3"),
    "M81": ("1D", "3"),
    "M82": ("1G", "3"),
    "M83": ("2K", "2L"),
    "M84": ("1H", "2J"),
    "M85": ("1B", "3"),
    "M86": ("1J", "2H"),
    "M87": ("1K", "3"),
    "M88": ("2D", "2G"),
}

THIRD_PLACE_SLOT_GROUPS = {
    "M74": {"A", "B", "C", "D", "F"},
    "M77": {"C", "D", "F", "G", "H"},
    "M79": {"C", "E", "F", "H", "I"},
    "M80": {"E", "H", "I", "J", "K"},
    "M81": {"B", "E", "F", "I", "J"},
    "M82": {"A", "E", "H", "I", "J"},
    "M85": {"E", "F", "G", "I", "J"},
    "M87": {"D", "E", "I", "J", "L"},
}

KNOCKOUT_RULES = {
    "M89": ("W73", "W75"),
    "M90": ("W74", "W77"),
    "M91": ("W76", "W78"),
    "M92": ("W79", "W80"),
    "M93": ("W83", "W84"),
    "M94": ("W81", "W82"),
    "M95": ("W86", "W88"),
    "M96": ("W85", "W87"),
    "M97": ("W89", "W90"),
    "M98": ("W93", "W94"),
    "M99": ("W91", "W92"),
    "M100": ("W95", "W96"),
    "M101": ("W97", "W98"),
    "M102": ("W99", "W100"),
    "M103": ("L101", "L102"),
    "M104": ("W101", "W102"),
}

In [22]:
def simulate_group_stage_for_official_bracket(
    group_matches,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    group_tables = defaultdict(dict)

    for _, match_row in group_matches.iterrows():
        group_name = str(match_row["group"])
        home_team = match_row["home_team"]
        away_team = match_row["away_team"]

        path_a.ensure_group_entry(group_tables, group_name, home_team)
        path_a.ensure_group_entry(group_tables, group_name, away_team)

        match = path_a.simulate_match(
            home_team=home_team,
            away_team=away_team,
            match_date=pd.Timestamp(match_row["match_date"]),
            neutral=int(match_row["neutral"]),
            tournament=match_row["tournament"],
            stage=match_row["stage"],
            group=group_name,
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
            knockout=False,
        )

        path_a.update_group_table(
            group_tables[group_name],
            match.home_team,
            match.away_team,
            match.home_goals,
            match.away_goals,
        )

    ranked_groups = {
        group_name: path_a.rank_group_table(table_entries, random_generator)
        for group_name, table_entries in group_tables.items()
    }

    group_winners = {}
    group_runners_up = {}
    third_places = []
    group_advancers = []

    for group_name, ranked_table in ranked_groups.items():
        if len(ranked_table) < 4:
            raise ValueError(f"Grupo {group_name} tem menos de 4 times.")

        winner = ranked_table[0]["team"]
        runner_up = ranked_table[1]["team"]

        group_winners[group_name] = winner
        group_runners_up[group_name] = runner_up

        group_advancers.extend([winner, runner_up])

        third_entry = ranked_table[2].copy()
        third_entry["group"] = group_name
        third_places.append(third_entry)

    best_third_places = path_a.rank_third_places(
        third_places,
        random_generator,
    )[:8]

    group_advancers.extend([entry["team"] for entry in best_third_places])

    return {
        "ranked_groups": ranked_groups,
        "group_winners": group_winners,
        "group_runners_up": group_runners_up,
        "best_third_places": best_third_places,
        "group_advancers": group_advancers,
    }

In [23]:
def resolve_third_place_slots(best_third_places):
    available_groups = {
        str(entry["group"]): entry["team"]
        for entry in best_third_places
    }

    group_priority = {
        str(entry["group"]): position
        for position, entry in enumerate(best_third_places)
    }

    slot_order = sorted(
        THIRD_PLACE_SLOT_GROUPS,
        key=lambda slot: len(
            THIRD_PLACE_SLOT_GROUPS[slot].intersection(available_groups)
        ),
    )

    def backtrack(slot_index, remaining_groups, current_assignment):
        if slot_index == len(slot_order):
            return current_assignment

        slot = slot_order[slot_index]

        eligible_groups = [
            group
            for group in remaining_groups
            if group in THIRD_PLACE_SLOT_GROUPS[slot]
        ]

        eligible_groups = sorted(
            eligible_groups,
            key=lambda group: group_priority[group],
        )

        for group in eligible_groups:
            next_assignment = dict(current_assignment)
            next_assignment[slot] = remaining_groups[group]

            next_remaining_groups = dict(remaining_groups)
            del next_remaining_groups[group]

            solved_assignment = backtrack(
                slot_index + 1,
                next_remaining_groups,
                next_assignment,
            )

            if solved_assignment is not None:
                return solved_assignment

        return None

    assignment = backtrack(
        slot_index=0,
        remaining_groups=available_groups,
        current_assignment={},
    )

    if assignment is None:
        raise RuntimeError(
            "Não foi possível encaixar os melhores terceiros nos slots oficiais."
        )

    return assignment

In [24]:
def resolve_group_position_token(
    token,
    group_winners,
    group_runners_up,
):
    position = token[0]
    group = token[1]

    if position == "1":
        return group_winners[group]

    if position == "2":
        return group_runners_up[group]

    raise ValueError(f"Token inválido: {token}")


def create_official_round_of_32_pairs(
    group_winners,
    group_runners_up,
    best_third_places,
):
    third_place_assignment = resolve_third_place_slots(best_third_places)

    match_pairs = {}

    for match_id, (home_token, away_token) in ROUND_OF_32_RULES.items():
        if home_token == "3":
            home_team = third_place_assignment[match_id]
        else:
            home_team = resolve_group_position_token(
                home_token,
                group_winners,
                group_runners_up,
            )

        if away_token == "3":
            away_team = third_place_assignment[match_id]
        else:
            away_team = resolve_group_position_token(
                away_token,
                group_winners,
                group_runners_up,
            )

        match_pairs[match_id] = (home_team, away_team)

    return match_pairs

In [25]:
def get_loser_from_match(home_team, away_team, winner):
    return away_team if winner == home_team else home_team


def resolve_knockout_rule(rule, match_results):
    result = match_results[f"M{rule[1:]}"]

    if rule.startswith("W"):
        return result.winner

    if rule.startswith("L"):
        return get_loser_from_match(
            result.home_team,
            result.away_team,
            result.winner,
        )

    raise ValueError(f"Regra inválida: {rule}")


def simulate_official_knockout_match(
    match_id,
    home_team,
    away_team,
    stage,
    match_date,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    match = path_a.simulate_match(
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        neutral=1,
        tournament="FIFA World Cup",
        stage=stage,
        group=None,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
        knockout=True,
    )

    if match.winner is None:
        raise RuntimeError(f"{match_id} terminou sem vencedor.")

    return match

In [26]:
def simulate_official_knockout_bracket_with_round_tracking(
    group_winners,
    group_runners_up,
    best_third_places,
    start_date,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    match_results = {}

    round_of_32_pairs = create_official_round_of_32_pairs(
        group_winners=group_winners,
        group_runners_up=group_runners_up,
        best_third_places=best_third_places,
    )

    round_of_32_teams = sorted(
        {team for pair in round_of_32_pairs.values() for team in pair}
    )

    round_of_16_teams = []
    quarterfinalists = []
    semifinalists = []
    finalists = []

    match_stage = {}
    for match_number in range(73, 89):
        match_stage[f"M{match_number}"] = "round_of_32"
    for match_number in range(89, 97):
        match_stage[f"M{match_number}"] = "round_of_16"
    for match_number in range(97, 101):
        match_stage[f"M{match_number}"] = "quarterfinal"
    for match_number in range(101, 103):
        match_stage[f"M{match_number}"] = "semifinal"
    match_stage["M103"] = "third_place"
    match_stage["M104"] = "final"

    for match_number in range(73, 89):
        match_id = f"M{match_number}"
        home_team, away_team = round_of_32_pairs[match_id]

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=match_number - 73),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    round_of_16_teams = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(73, 89)
    )

    for match_number in range(89, 97):
        match_id = f"M{match_number}"
        home_rule, away_rule = KNOCKOUT_RULES[match_id]

        home_team = resolve_knockout_rule(home_rule, match_results)
        away_team = resolve_knockout_rule(away_rule, match_results)

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=20 + match_number - 89),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    quarterfinalists = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(89, 97)
    )

    for match_number in range(97, 101):
        match_id = f"M{match_number}"
        home_rule, away_rule = KNOCKOUT_RULES[match_id]

        home_team = resolve_knockout_rule(home_rule, match_results)
        away_team = resolve_knockout_rule(away_rule, match_results)

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=32 + match_number - 97),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    semifinalists = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(97, 101)
    )

    for match_number in range(101, 103):
        match_id = f"M{match_number}"
        home_rule, away_rule = KNOCKOUT_RULES[match_id]

        home_team = resolve_knockout_rule(home_rule, match_results)
        away_team = resolve_knockout_rule(away_rule, match_results)

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=42 + match_number - 101),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    finalists = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(101, 103)
    )

    third_home = resolve_knockout_rule("L101", match_results)
    third_away = resolve_knockout_rule("L102", match_results)

    match_results["M103"] = simulate_official_knockout_match(
        match_id="M103",
        home_team=third_home,
        away_team=third_away,
        stage="third_place",
        match_date=start_date + pd.Timedelta(days=49),
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    final_home = resolve_knockout_rule("W101", match_results)
    final_away = resolve_knockout_rule("W102", match_results)

    match_results["M104"] = simulate_official_knockout_match(
        match_id="M104",
        home_team=final_home,
        away_team=final_away,
        stage="final",
        match_date=start_date + pd.Timedelta(days=50),
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    final_match = match_results["M104"]
    third_place_match = match_results["M103"]

    champion = final_match.winner
    runner_up = get_loser_from_match(
        final_match.home_team,
        final_match.away_team,
        final_match.winner,
    )

    third_place = third_place_match.winner
    fourth_place = get_loser_from_match(
        third_place_match.home_team,
        third_place_match.away_team,
        third_place_match.winner,
    )

    return {
        "champion": champion,
        "runner_up": runner_up,
        "third_place": third_place,
        "fourth_place": fourth_place,
        "finalists": finalists,
        "semifinalists": semifinalists,
        "quarterfinalists": quarterfinalists,
        "round_of_16_teams": round_of_16_teams,
        "round_of_32_teams": round_of_32_teams,
        "match_results": match_results,
    }

In [27]:
@dataclass
class TournamentSimulationOfficialBracket:
    champion: str
    runner_up: str
    third_place: str
    fourth_place: str
    finalists: list[str]
    semifinalists: list[str]
    quarterfinalists: list[str]
    round_of_16_teams: list[str]
    round_of_32_teams: list[str]
    group_advancers: list[str]
    playoff_qualifiers: list[str]

In [28]:
def run_one_simulation_official_bracket(
    schedule_data,
    playoff_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    states = copy.deepcopy(initial_states)

    slots, playoff_qualifiers = path_a.simulate_playoffs(
        playoff_data=playoff_data,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    resolved_schedule = path_a.resolve_schedule_slots(schedule_data, slots)

    group_matches = resolved_schedule[
        resolved_schedule["stage"].str.lower().str.contains("group")
    ].copy()

    if group_matches.empty:
        raise ValueError("A tabela da Copa não tem jogos de fase de grupos.")

    group_result = simulate_group_stage_for_official_bracket(
        group_matches=group_matches,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    knockout_start_date = group_matches["match_date"].max() + pd.Timedelta(days=1)

    knockout_result = simulate_official_knockout_bracket_with_round_tracking(
        group_winners=group_result["group_winners"],
        group_runners_up=group_result["group_runners_up"],
        best_third_places=group_result["best_third_places"],
        start_date=knockout_start_date,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    return TournamentSimulationOfficialBracket(
        champion=knockout_result["champion"],
        runner_up=knockout_result["runner_up"],
        third_place=knockout_result["third_place"],
        fourth_place=knockout_result["fourth_place"],
        finalists=knockout_result["finalists"],
        semifinalists=knockout_result["semifinalists"],
        quarterfinalists=knockout_result["quarterfinalists"],
        round_of_16_teams=knockout_result["round_of_16_teams"],
        round_of_32_teams=knockout_result["round_of_32_teams"],
        group_advancers=group_result["group_advancers"],
        playoff_qualifiers=playoff_qualifiers,
    )

In [29]:
def run_simulations_official_bracket(
    schedule_data,
    playoff_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    simulations,
    seed,
):
    random_generator = path_a.default_rng(seed)

    champion_counter = Counter()
    runner_up_counter = Counter()
    third_place_counter = Counter()
    fourth_place_counter = Counter()
    finalist_counter = Counter()
    semifinal_counter = Counter()
    quarterfinal_counter = Counter()
    round_of_16_counter = Counter()
    round_of_32_counter = Counter()
    group_advancement_counter = Counter()
    playoff_qualification_counter = Counter()
    champion_given_playoff_counter = Counter()

    for _ in range(simulations):
        simulation = run_one_simulation_official_bracket(
            schedule_data=schedule_data,
            playoff_data=playoff_data,
            initial_states=initial_states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

        champion_counter[simulation.champion] += 1
        runner_up_counter[simulation.runner_up] += 1
        third_place_counter[simulation.third_place] += 1
        fourth_place_counter[simulation.fourth_place] += 1

        for team in simulation.finalists:
            finalist_counter[team] += 1

        for team in simulation.semifinalists:
            semifinal_counter[team] += 1

        for team in simulation.quarterfinalists:
            quarterfinal_counter[team] += 1

        for team in simulation.round_of_16_teams:
            round_of_16_counter[team] += 1

        for team in simulation.round_of_32_teams:
            round_of_32_counter[team] += 1

        for team in simulation.group_advancers:
            group_advancement_counter[team] += 1

        for team in simulation.playoff_qualifiers:
            playoff_qualification_counter[team] += 1

        if simulation.champion in simulation.playoff_qualifiers:
            champion_given_playoff_counter[simulation.champion] += 1

    return {
        "champion": champion_counter,
        "runner_up": runner_up_counter,
        "third_place": third_place_counter,
        "fourth_place": fourth_place_counter,
        "final": finalist_counter,
        "semifinal": semifinal_counter,
        "quarterfinal": quarterfinal_counter,
        "round_of_16": round_of_16_counter,
        "round_of_32": round_of_32_counter,
        "group_advancement": group_advancement_counter,
        "playoff_qualification": playoff_qualification_counter,
        "champion_given_playoff": champion_given_playoff_counter,
        "simulations": simulations,
    }

In [30]:
def merge_simulation_results(results_list):
    counter_keys = [
        "champion",
        "runner_up",
        "third_place",
        "fourth_place",
        "final",
        "semifinal",
        "quarterfinal",
        "round_of_16",
        "round_of_32",
        "group_advancement",
        "playoff_qualification",
        "champion_given_playoff",
    ]

    merged_results = {
        key: Counter()
        for key in counter_keys
    }

    total_simulations = 0

    for result in results_list:
        total_simulations += int(result["simulations"])

        for key in counter_keys:
            if key in result:
                merged_results[key].update(result[key])

    merged_results["simulations"] = total_simulations

    return merged_results

In [31]:
def run_simulation_chunk(
    chunk_id,
    simulations_in_chunk,
    base_seed,
    schedule_data,
    playoff_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
):
    path_a.FEATURE_COLUMNS = FEATURE_COLUMNS_USED

    # Se você estiver usando o ajuste de mercado via odds, mantenha esta linha.
    # Se não estiver usando o blend com odds, pode comentar.
    path_a.simulate_match = simulate_match_with_market_blend

    chunk_seed = int(base_seed + 1000003 * chunk_id)

    result = run_simulations_official_bracket(
        schedule_data=schedule_data,
        playoff_data=playoff_data,
        initial_states=initial_states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        simulations=simulations_in_chunk,
        seed=chunk_seed,
    )

    return result

In [32]:
TOTAL_SIMULATIONS = 100_000
BASE_SEED = 42

N_JOBS = max(1, os.cpu_count() - 1)
N_CHUNKS = N_JOBS * 4

chunk_sizes = split_simulations(
    total_simulations=TOTAL_SIMULATIONS,
    n_chunks=N_CHUNKS,
)

print("CPUs disponíveis:", os.cpu_count())
print("N_JOBS:", N_JOBS)
print("N_CHUNKS:", len(chunk_sizes))
print("Primeiros tamanhos dos blocos:", chunk_sizes[:10])
print("Total:", sum(chunk_sizes))

CPUs disponíveis: 12
N_JOBS: 11
N_CHUNKS: 44
Primeiros tamanhos dos blocos: [2273, 2273, 2273, 2273, 2273, 2273, 2273, 2273, 2273, 2273]
Total: 100000


In [33]:
def get_match_from_schedule(schedule_data, home_team, away_team):
    same_order = schedule_data[
        (schedule_data["home_team"] == home_team)
        & (schedule_data["away_team"] == away_team)
    ]

    reverse_order = schedule_data[
        (schedule_data["home_team"] == away_team)
        & (schedule_data["away_team"] == home_team)
    ]

    if not same_order.empty:
        match_row = same_order.iloc[0].copy()
        return match_row, False

    if not reverse_order.empty:
        match_row = reverse_order.iloc[0].copy()
        return match_row, True

    raise ValueError(f"Jogo não encontrado na tabela: {home_team} x {away_team}")


def adjust_score_matrix_to_target_probabilities(
    score_matrix,
    target_probabilities,
):
    model_outcome_probabilities = {
        "home_win": 0.0,
        "draw": 0.0,
        "away_win": 0.0,
    }

    max_home_goals, max_away_goals = score_matrix.shape

    for home_goals in range(max_home_goals):
        for away_goals in range(max_away_goals):
            outcome = get_score_outcome(home_goals, away_goals)
            model_outcome_probabilities[outcome] += score_matrix[
                home_goals,
                away_goals,
            ]

    adjusted_matrix = score_matrix.copy()

    for home_goals in range(max_home_goals):
        for away_goals in range(max_away_goals):
            outcome = get_score_outcome(home_goals, away_goals)
            model_probability = model_outcome_probabilities[outcome]

            if model_probability > 0:
                adjustment_factor = (
                    target_probabilities[outcome] / model_probability
                )
                adjusted_matrix[home_goals, away_goals] *= adjustment_factor

    return adjusted_matrix / adjusted_matrix.sum()


def get_top_scorelines(score_matrix, home_team, away_team, top_n=15):
    rows = []

    for home_goals in range(score_matrix.shape[0]):
        for away_goals in range(score_matrix.shape[1]):
            rows.append(
                {
                    "score": f"{home_team} {home_goals} x {away_goals} {away_team}",
                    "home_goals": home_goals,
                    "away_goals": away_goals,
                    "probability": score_matrix[home_goals, away_goals],
                }
            )

    scorelines_data = pd.DataFrame(rows)

    return scorelines_data.sort_values(
        "probability",
        ascending=False,
    ).head(top_n).reset_index(drop=True)

In [34]:
def analyze_specific_match(
    home_team,
    away_team,
    schedule_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    market_weight=0.35,
    max_goals=10,
    simulations=100_000,
    seed=42,
):
    match_row, reversed_order = get_match_from_schedule(
        schedule_data=schedule_data,
        home_team=home_team,
        away_team=away_team,
    )

    model_home_team = match_row["home_team"]
    model_away_team = match_row["away_team"]

    match_date = pd.Timestamp(match_row["match_date"])

    market_probabilities = odds_lookup.get(
        match_date,
        model_home_team,
        model_away_team,
    )

    feature_row = path_a.build_feature_row(
        home_team=model_home_team,
        away_team=model_away_team,
        match_date=match_date,
        neutral=int(match_row["neutral"]),
        tournament=match_row["tournament"],
        states=initial_states,
        ranking_lookup=ranking_lookup,
        market_probabilities=market_probabilities,
    )

    home_lambda, away_lambda = path_a.predict_expected_goals(
        home_model,
        away_model,
        feature_row,
    )

    original_score_matrix = calculate_score_probability_matrix(
        home_lambda=home_lambda,
        away_lambda=away_lambda,
        max_goals=max_goals,
    )

    model_probabilities = calculate_outcome_probabilities_from_score_matrix(
        original_score_matrix
    )

    target_probabilities = blend_outcome_probabilities(
        model_probabilities=model_probabilities,
        market_probabilities=market_probabilities,
        market_weight=market_weight,
    )

    adjusted_score_matrix = adjust_score_matrix_to_target_probabilities(
        score_matrix=original_score_matrix,
        target_probabilities=target_probabilities,
    )

    adjusted_probabilities = calculate_outcome_probabilities_from_score_matrix(
        adjusted_score_matrix
    )

    top_scorelines_exact = get_top_scorelines(
        score_matrix=adjusted_score_matrix,
        home_team=model_home_team,
        away_team=model_away_team,
        top_n=20,
    )

    rng = np.random.default_rng(seed)

    flat_probabilities = adjusted_score_matrix.ravel()

    sampled_indices = rng.choice(
        len(flat_probabilities),
        size=simulations,
        replace=True,
        p=flat_probabilities,
    )

    sampled_scores = [
        np.unravel_index(index, adjusted_score_matrix.shape)
        for index in sampled_indices
    ]

    score_counter = Counter(sampled_scores)

    simulated_scorelines = pd.DataFrame(
        [
            {
                "score": f"{model_home_team} {home_goals} x {away_goals} {model_away_team}",
                "home_goals": home_goals,
                "away_goals": away_goals,
                "count": count,
                "simulated_probability": count / simulations,
            }
            for (home_goals, away_goals), count in score_counter.most_common()
        ]
    )

    simulated_outcomes = {
        "home_win": 0,
        "draw": 0,
        "away_win": 0,
    }

    for (home_goals, away_goals), count in score_counter.items():
        outcome = get_score_outcome(home_goals, away_goals)
        simulated_outcomes[outcome] += count

    probabilities_summary = pd.DataFrame(
        [
            {
                "outcome": f"{model_home_team} vence",
                "model_probability": model_probabilities["home_win"],
                "market_probability": market_probabilities["market_home_prob"],
                "final_probability": adjusted_probabilities["home_win"],
                "simulated_probability": simulated_outcomes["home_win"] / simulations,
            },
            {
                "outcome": "Empate",
                "model_probability": model_probabilities["draw"],
                "market_probability": market_probabilities["market_draw_prob"],
                "final_probability": adjusted_probabilities["draw"],
                "simulated_probability": simulated_outcomes["draw"] / simulations,
            },
            {
                "outcome": f"{model_away_team} vence",
                "model_probability": model_probabilities["away_win"],
                "market_probability": market_probabilities["market_away_prob"],
                "final_probability": adjusted_probabilities["away_win"],
                "simulated_probability": simulated_outcomes["away_win"] / simulations,
            },
        ]
    )

    lambdas_data = pd.DataFrame(
        [
            {
                "home_team": model_home_team,
                "away_team": model_away_team,
                "home_lambda": home_lambda,
                "away_lambda": away_lambda,
                "market_weight": market_weight,
                "simulations": simulations,
                "match_date": match_date,
            }
        ]
    )

    return {
        "match": match_row,
        "lambdas": lambdas_data,
        "probabilities": probabilities_summary,
        "top_scorelines_exact": top_scorelines_exact,
        "top_scorelines_simulated": simulated_scorelines.head(20),
        "adjusted_score_matrix": adjusted_score_matrix,
    }

In [35]:
match_analysis = analyze_specific_match(
    home_team="Mexico",
    away_team="South Africa",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Mexico,South Africa,1.784491,0.379169,0.4,100000,2026-06-11


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Mexico vence,0.721456,0.677995,0.704072,0.70614
1,Empate,0.206834,0.212438,0.209076,0.20795
2,South Africa vence,0.071710,0.109567,0.086853,0.08591


,score,home_goals,away_goals,probability
0,Mexico 1 x 0 South Africa,1,0,0.200104
1,Mexico 2 x 0 South Africa,2,0,0.178542
2,Mexico 0 x 0 South Africa,0,0,0.116149
3,Mexico 3 x 0 South Africa,3,0,0.106202
4,Mexico 1 x 1 South Africa,1,1,0.078590
5,Mexico 2 x 1 South Africa,2,1,0.067698
6,Mexico 0 x 1 South Africa,0,1,0.052768
7,Mexico 4 x 0 South Africa,4,0,0.047379
8,Mexico 3 x 1 South Africa,3,1,0.040269
9,Mexico 4 x 1 South Africa,4,1,0.017965


,score,home_goals,away_goals,count,simulated_probability
0,Mexico 1 x 0 South Africa,1,0,19936,0.19936
1,Mexico 2 x 0 South Africa,2,0,18123,0.18123
2,Mexico 0 x 0 South Africa,0,0,11563,0.11563
3,Mexico 3 x 0 South Africa,3,0,10606,0.10606
4,Mexico 1 x 1 South Africa,1,1,7803,0.07803
5,Mexico 2 x 1 South Africa,2,1,6812,0.06812
6,Mexico 0 x 1 South Africa,0,1,5289,0.05289
7,Mexico 4 x 0 South Africa,4,0,4671,0.04671
8,Mexico 3 x 1 South Africa,3,1,4014,0.04014
9,Mexico 4 x 1 South Africa,4,1,1819,0.01819


In [36]:
match_analysis = analyze_specific_match(
    home_team="Czechia",
    away_team="South Korea",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,South Korea,Czechia,1.345521,0.798322,0.4,100000,2026-06-11


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,South Korea vence,0.496665,0.354819,0.439927,0.43988
1,Empate,0.281225,0.309036,0.292349,0.29144
2,Czechia vence,0.222111,0.336145,0.267724,0.26868


,score,home_goals,away_goals,probability
0,South Korea 1 x 0 Czechia,1,0,0.139685
1,South Korea 1 x 1 Czechia,1,1,0.130875
2,South Korea 0 x 0 Czechia,0,0,0.121840
3,South Korea 0 x 1 Czechia,0,1,0.112781
4,South Korea 2 x 0 Czechia,2,0,0.093974
5,South Korea 2 x 1 Czechia,2,1,0.075022
6,South Korea 1 x 2 Czechia,1,2,0.060573
7,South Korea 0 x 2 Czechia,0,2,0.045018
8,South Korea 3 x 0 Czechia,3,0,0.042148
9,South Korea 2 x 2 Czechia,2,2,0.035145


,score,home_goals,away_goals,count,simulated_probability
0,South Korea 1 x 0 Czechia,1,0,13831,0.13831
1,South Korea 1 x 1 Czechia,1,1,13141,0.13141
2,South Korea 0 x 0 Czechia,0,0,12121,0.12121
3,South Korea 0 x 1 Czechia,0,1,11244,0.11244
4,South Korea 2 x 0 Czechia,2,0,9446,0.09446
5,South Korea 2 x 1 Czechia,2,1,7572,0.07572
6,South Korea 1 x 2 Czechia,1,2,6077,0.06077
7,South Korea 0 x 2 Czechia,0,2,4587,0.04587
8,South Korea 3 x 0 Czechia,3,0,4177,0.04177
9,South Korea 2 x 2 Czechia,2,2,3442,0.03442


In [37]:
match_analysis = analyze_specific_match(
    home_team="Canada",
    away_team="Bosnia and Herzegovina",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Canada,Bosnia and Herzegovina,2.310625,0.557265,0.4,100000,2026-06-12


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Canada vence,0.770041,0.529694,0.673902,0.67579
1,Empate,0.157191,0.266318,0.200842,0.20039
2,Bosnia and Herzegovina vence,0.072768,0.203988,0.125256,0.12382


,score,home_goals,away_goals,probability
0,Canada 2 x 0 Bosnia and Herzegovina,2,0,0.132744
1,Canada 1 x 0 Bosnia and Herzegovina,1,0,0.114899
2,Canada 3 x 0 Bosnia and Herzegovina,3,0,0.102241
3,Canada 1 x 1 Bosnia and Herzegovina,1,1,0.093481
4,Canada 2 x 1 Bosnia and Herzegovina,2,1,0.073974
5,Canada 0 x 0 Bosnia and Herzegovina,0,0,0.072599
6,Canada 4 x 0 Bosnia and Herzegovina,4,0,0.059060
7,Canada 3 x 1 Bosnia and Herzegovina,3,1,0.056975
8,Canada 0 x 1 Bosnia and Herzegovina,0,1,0.054504
9,Canada 1 x 2 Bosnia and Herzegovina,1,2,0.035090


,score,home_goals,away_goals,count,simulated_probability
0,Canada 2 x 0 Bosnia and Herzegovina,2,0,13329,0.13329
1,Canada 1 x 0 Bosnia and Herzegovina,1,0,11538,0.11538
2,Canada 3 x 0 Bosnia and Herzegovina,3,0,10263,0.10263
3,Canada 1 x 1 Bosnia and Herzegovina,1,1,9274,0.09274
4,Canada 2 x 1 Bosnia and Herzegovina,2,1,7416,0.07416
5,Canada 0 x 0 Bosnia and Herzegovina,0,0,7227,0.07227
6,Canada 4 x 0 Bosnia and Herzegovina,4,0,5867,0.05867
7,Canada 3 x 1 Bosnia and Herzegovina,3,1,5770,0.05770
8,Canada 0 x 1 Bosnia and Herzegovina,0,1,5447,0.05447
9,Canada 1 x 2 Bosnia and Herzegovina,1,2,3417,0.03417


In [38]:
match_analysis = analyze_specific_match(
    home_team="United States",
    away_team="Paraguay",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,United States,Paraguay,1.37071,0.922931,0.4,100000,2026-06-12


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,United States vence,0.472197,0.492007,0.480121,0.48071
1,Empate,0.275058,0.280733,0.277328,0.27655
2,Paraguay vence,0.252744,0.227260,0.242551,0.24274


,score,home_goals,away_goals,probability
0,United States 1 x 0 Paraguay,1,0,0.140623
1,United States 1 x 1 Paraguay,1,1,0.128697
2,United States 0 x 0 Paraguay,0,0,0.101731
3,United States 2 x 0 Paraguay,2,0,0.096377
4,United States 0 x 1 Paraguay,0,1,0.089366
5,United States 2 x 1 Paraguay,2,1,0.088949
6,United States 1 x 2 Paraguay,1,2,0.056527
7,United States 3 x 0 Paraguay,3,0,0.044035
8,United States 0 x 2 Paraguay,0,2,0.041240
9,United States 2 x 2 Paraguay,2,2,0.040703


,score,home_goals,away_goals,count,simulated_probability
0,United States 1 x 0 Paraguay,1,0,13898,0.13898
1,United States 1 x 1 Paraguay,1,1,12887,0.12887
2,United States 0 x 0 Paraguay,0,0,10096,0.10096
3,United States 2 x 0 Paraguay,2,0,9835,0.09835
4,United States 0 x 1 Paraguay,0,1,8944,0.08944
5,United States 2 x 1 Paraguay,2,1,8941,0.08941
6,United States 1 x 2 Paraguay,1,2,5706,0.05706
7,United States 3 x 0 Paraguay,3,0,4381,0.04381
8,United States 0 x 2 Paraguay,0,2,4104,0.04104
9,United States 2 x 2 Paraguay,2,2,4073,0.04073


In [39]:
match_analysis = analyze_specific_match(
    home_team="Australia",
    away_team="Turkey",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Australia,Turkey,1.279459,1.315152,0.4,100000,2026-06-13


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Australia vence,0.359539,0.185091,0.289760,0.28851
1,Empate,0.264184,0.254192,0.260187,0.25987
2,Turkey vence,0.376277,0.560717,0.450053,0.45162


,score,home_goals,away_goals,probability
0,Australia 1 x 1 Turkey,1,1,0.123753
1,Australia 0 x 1 Turkey,0,1,0.117464
2,Australia 1 x 2 Turkey,1,2,0.098828
3,Australia 0 x 2 Turkey,0,2,0.077242
4,Australia 1 x 0 Turkey,1,0,0.077000
5,Australia 0 x 0 Turkey,0,0,0.073545
6,Australia 2 x 1 Turkey,2,1,0.064784
7,Australia 2 x 2 Turkey,2,2,0.052059
8,Australia 2 x 0 Turkey,2,0,0.049259
9,Australia 1 x 3 Turkey,1,3,0.043325


,score,home_goals,away_goals,count,simulated_probability
0,Australia 1 x 1 Turkey,1,1,12399,0.12399
1,Australia 0 x 1 Turkey,0,1,11726,0.11726
2,Australia 1 x 2 Turkey,1,2,9930,0.09930
3,Australia 0 x 2 Turkey,0,2,7790,0.07790
4,Australia 1 x 0 Turkey,1,0,7596,0.07596
5,Australia 0 x 0 Turkey,0,0,7308,0.07308
6,Australia 2 x 1 Turkey,2,1,6478,0.06478
7,Australia 2 x 2 Turkey,2,2,5180,0.05180
8,Australia 2 x 0 Turkey,2,0,4967,0.04967
9,Australia 1 x 3 Turkey,1,3,4510,0.04510


In [40]:
match_analysis = analyze_specific_match(
    home_team="Qatar",
    away_team="Switzerland",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Qatar,Switzerland,0.609022,2.341314,0.4,100000,2026-06-13


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Qatar vence,0.079124,0.071405,0.076036,0.07573
1,Empate,0.158141,0.140165,0.150951,0.15271
2,Switzerland vence,0.762735,0.788430,0.773013,0.77156


,score,home_goals,away_goals,probability
0,Qatar 0 x 2 Switzerland,0,2,0.145346
1,Qatar 0 x 1 Switzerland,0,1,0.124158
2,Qatar 0 x 3 Switzerland,0,3,0.113434
3,Qatar 1 x 2 Switzerland,1,2,0.088519
4,Qatar 1 x 1 Switzerland,1,1,0.071217
5,Qatar 1 x 3 Switzerland,1,3,0.069083
6,Qatar 0 x 4 Switzerland,0,4,0.066396
7,Qatar 0 x 0 Switzerland,0,0,0.049945
8,Qatar 1 x 4 Switzerland,1,4,0.040437
9,Qatar 0 x 5 Switzerland,0,5,0.031091


,score,home_goals,away_goals,count,simulated_probability
0,Qatar 0 x 2 Switzerland,0,2,14462,0.14462
1,Qatar 0 x 1 Switzerland,0,1,12336,0.12336
2,Qatar 0 x 3 Switzerland,0,3,11290,0.11290
3,Qatar 1 x 2 Switzerland,1,2,8899,0.08899
4,Qatar 1 x 1 Switzerland,1,1,7246,0.07246
5,Qatar 1 x 3 Switzerland,1,3,6974,0.06974
6,Qatar 0 x 4 Switzerland,0,4,6537,0.06537
7,Qatar 0 x 0 Switzerland,0,0,5042,0.05042
8,Qatar 1 x 4 Switzerland,1,4,3965,0.03965
9,Qatar 0 x 5 Switzerland,0,5,3201,0.03201


In [41]:
match_analysis = analyze_specific_match(
    home_team="Brazil",
    away_team="Morocco",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Brazil,Morocco,1.701891,0.982022,0.4,100000,2026-06-13


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Brazil vence,0.542785,0.576631,0.556323,0.55917
1,Empate,0.239975,0.250379,0.244137,0.24176
2,Morocco vence,0.217241,0.172989,0.199540,0.19907


,score,home_goals,away_goals,probability
0,Brazil 1 x 0 Morocco,1,0,0.119131
1,Brazil 1 x 1 Morocco,1,1,0.116121
2,Brazil 2 x 0 Morocco,2,0,0.101374
3,Brazil 2 x 1 Morocco,2,1,0.099551
4,Brazil 0 x 0 Morocco,0,0,0.069480
5,Brazil 0 x 1 Morocco,0,1,0.061603
6,Brazil 3 x 0 Morocco,3,0,0.057509
7,Brazil 3 x 1 Morocco,3,1,0.056475
8,Brazil 1 x 2 Morocco,1,2,0.051478
9,Brazil 2 x 2 Morocco,2,2,0.048518


,score,home_goals,away_goals,count,simulated_probability
0,Brazil 1 x 0 Morocco,1,0,11921,0.11921
1,Brazil 1 x 1 Morocco,1,1,11409,0.11409
2,Brazil 2 x 0 Morocco,2,0,10207,0.10207
3,Brazil 2 x 1 Morocco,2,1,10158,0.10158
4,Brazil 0 x 0 Morocco,0,0,6933,0.06933
5,Brazil 0 x 1 Morocco,0,1,6138,0.06138
6,Brazil 3 x 0 Morocco,3,0,5817,0.05817
7,Brazil 3 x 1 Morocco,3,1,5560,0.05560
8,Brazil 1 x 2 Morocco,1,2,5182,0.05182
9,Brazil 2 x 2 Morocco,2,2,4852,0.04852


In [42]:
match_analysis = analyze_specific_match(
    home_team="Haiti",
    away_team="Scotland",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Haiti,Scotland,0.878888,1.797605,0.4,100000,2026-06-13


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Haiti vence,0.180427,0.158305,0.171578,0.17046
1,Empate,0.228773,0.220891,0.225620,0.22633
2,Scotland vence,0.590801,0.620804,0.602802,0.60321


,score,home_goals,away_goals,probability
0,Haiti 0 x 1 Scotland,0,1,0.126195
1,Haiti 0 x 2 Scotland,0,2,0.113425
2,Haiti 1 x 1 Scotland,1,1,0.107205
3,Haiti 1 x 2 Scotland,1,2,0.099688
4,Haiti 0 x 3 Scotland,0,3,0.067964
5,Haiti 0 x 0 Scotland,0,0,0.067856
6,Haiti 1 x 3 Scotland,1,3,0.059733
7,Haiti 1 x 0 Scotland,1,0,0.057506
8,Haiti 2 x 1 Scotland,2,1,0.045426
9,Haiti 2 x 2 Scotland,2,2,0.042343


,score,home_goals,away_goals,count,simulated_probability
0,Haiti 0 x 1 Scotland,0,1,12542,0.12542
1,Haiti 0 x 2 Scotland,0,2,11379,0.11379
2,Haiti 1 x 1 Scotland,1,1,10812,0.10812
3,Haiti 1 x 2 Scotland,1,2,10169,0.10169
4,Haiti 0 x 0 Scotland,0,0,6762,0.06762
5,Haiti 0 x 3 Scotland,0,3,6718,0.06718
6,Haiti 1 x 3 Scotland,1,3,5978,0.05978
7,Haiti 1 x 0 Scotland,1,0,5667,0.05667
8,Haiti 2 x 1 Scotland,2,1,4405,0.04405
9,Haiti 2 x 2 Scotland,2,2,4236,0.04236


In [43]:
match_analysis = analyze_specific_match(
    home_team="Germany",
    away_team="Curacao",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Germany,Curacao,3.273279,0.581693,0.4,100000,2026-06-14


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Germany vence,0.878518,0.925595,0.897349,0.89767
1,Empate,0.085315,0.053479,0.072580,0.07235
2,Curacao vence,0.036167,0.020926,0.030071,0.02998


,score,home_goals,away_goals,probability
0,Germany 3 x 0 Curacao,3,0,0.126495
1,Germany 2 x 0 Curacao,2,0,0.115935
2,Germany 4 x 0 Curacao,4,0,0.103514
3,Germany 3 x 1 Curacao,3,1,0.073581
4,Germany 1 x 0 Curacao,1,0,0.070837
5,Germany 5 x 0 Curacao,5,0,0.067766
6,Germany 2 x 1 Curacao,2,1,0.067438
7,Germany 4 x 1 Curacao,4,1,0.060213
8,Germany 5 x 1 Curacao,5,1,0.039419
9,Germany 6 x 0 Curacao,6,0,0.036969


,score,home_goals,away_goals,count,simulated_probability
0,Germany 3 x 0 Curacao,3,0,12509,0.12509
1,Germany 2 x 0 Curacao,2,0,11660,0.11660
2,Germany 4 x 0 Curacao,4,0,10528,0.10528
3,Germany 3 x 1 Curacao,3,1,7487,0.07487
4,Germany 1 x 0 Curacao,1,0,6979,0.06979
5,Germany 5 x 0 Curacao,5,0,6792,0.06792
6,Germany 2 x 1 Curacao,2,1,6668,0.06668
7,Germany 4 x 1 Curacao,4,1,6038,0.06038
8,Germany 5 x 1 Curacao,5,1,3904,0.03904
9,Germany 6 x 0 Curacao,6,0,3696,0.03696


In [44]:
match_analysis = analyze_specific_match(
    home_team="Netherlands",
    away_team="Japan",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Netherlands,Japan,1.456146,1.171899,0.4,100000,2026-06-14


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Netherlands vence,0.436485,0.475797,0.452210,0.45229
1,Empate,0.259143,0.268054,0.262707,0.26179
2,Japan vence,0.304372,0.256149,0.285083,0.28592


,score,home_goals,away_goals,probability
0,Netherlands 1 x 1 Japan,1,1,0.124935
1,Netherlands 1 x 0 Japan,1,0,0.108951
2,Netherlands 2 x 1 Japan,2,1,0.092960
3,Netherlands 2 x 0 Japan,2,0,0.079324
4,Netherlands 0 x 1 Japan,0,1,0.079270
5,Netherlands 0 x 0 Japan,0,0,0.073213
6,Netherlands 1 x 2 Japan,1,2,0.067636
7,Netherlands 2 x 2 Japan,2,2,0.053299
8,Netherlands 0 x 2 Japan,0,2,0.046448
9,Netherlands 3 x 1 Japan,3,1,0.045121


,score,home_goals,away_goals,count,simulated_probability
0,Netherlands 1 x 1 Japan,1,1,12422,0.12422
1,Netherlands 1 x 0 Japan,1,0,10857,0.10857
2,Netherlands 2 x 1 Japan,2,1,9350,0.09350
3,Netherlands 2 x 0 Japan,2,0,8016,0.08016
4,Netherlands 0 x 1 Japan,0,1,7874,0.07874
5,Netherlands 0 x 0 Japan,0,0,7288,0.07288
6,Netherlands 1 x 2 Japan,1,2,6755,0.06755
7,Netherlands 2 x 2 Japan,2,2,5354,0.05354
8,Netherlands 0 x 2 Japan,0,2,4649,0.04649
9,Netherlands 3 x 1 Japan,3,1,4487,0.04487


In [45]:
match_analysis = analyze_specific_match(
    home_team="Ecuador",
    away_team="Ivory Coast",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Ivory Coast,Ecuador,0.850331,1.464948,0.4,100000,2026-06-14


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Ivory Coast vence,0.218297,0.265992,0.237375,0.23758
1,Empate,0.265777,0.341040,0.295882,0.29697
2,Ecuador vence,0.515927,0.392968,0.466743,0.46545


,score,home_goals,away_goals,probability
0,Ivory Coast 1 x 1 Ecuador,1,1,0.136930
1,Ivory Coast 0 x 1 Ecuador,0,1,0.130858
2,Ivory Coast 0 x 0 Ecuador,0,0,0.109923
3,Ivory Coast 0 x 2 Ecuador,0,2,0.095850
4,Ivory Coast 1 x 0 Ecuador,1,0,0.091298
5,Ivory Coast 1 x 2 Ecuador,1,2,0.081504
6,Ivory Coast 2 x 1 Ecuador,2,1,0.056865
7,Ivory Coast 0 x 3 Ecuador,0,3,0.046805
8,Ivory Coast 2 x 2 Ecuador,2,2,0.042643
9,Ivory Coast 1 x 3 Ecuador,1,3,0.039800


,score,home_goals,away_goals,count,simulated_probability
0,Ivory Coast 1 x 1 Ecuador,1,1,13882,0.13882
1,Ivory Coast 0 x 1 Ecuador,0,1,13049,0.13049
2,Ivory Coast 0 x 0 Ecuador,0,0,10932,0.10932
3,Ivory Coast 0 x 2 Ecuador,0,2,9576,0.09576
4,Ivory Coast 1 x 0 Ecuador,1,0,9125,0.09125
5,Ivory Coast 1 x 2 Ecuador,1,2,8202,0.08202
6,Ivory Coast 2 x 1 Ecuador,2,1,5591,0.05591
7,Ivory Coast 0 x 3 Ecuador,0,3,4649,0.04649
8,Ivory Coast 2 x 2 Ecuador,2,2,4223,0.04223
9,Ivory Coast 1 x 3 Ecuador,1,3,3998,0.03998


In [46]:
match_analysis = analyze_specific_match(
    home_team="Sweden",
    away_team="Tunisia",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Sweden,Tunisia,1.417318,0.979078,0.4,100000,2026-06-14


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Sweden vence,0.471161,0.509259,0.48640,0.48756
1,Empate,0.268594,0.277509,0.27216,0.27075
2,Tunisia vence,0.260245,0.213231,0.24144,0.24169


,score,home_goals,away_goals,probability
0,Sweden 1 x 0 Tunisia,1,0,0.133214
1,Sweden 1 x 1 Tunisia,1,1,0.128018
2,Sweden 2 x 0 Tunisia,2,0,0.094403
3,Sweden 2 x 1 Tunisia,2,1,0.092428
4,Sweden 0 x 0 Tunisia,0,0,0.092254
5,Sweden 0 x 1 Tunisia,0,1,0.082699
6,Sweden 1 x 2 Tunisia,1,2,0.057379
7,Sweden 3 x 0 Tunisia,3,0,0.044600
8,Sweden 2 x 2 Tunisia,2,2,0.044412
9,Sweden 3 x 1 Tunisia,3,1,0.043667


,score,home_goals,away_goals,count,simulated_probability
0,Sweden 1 x 0 Tunisia,1,0,13307,0.13307
1,Sweden 1 x 1 Tunisia,1,1,12629,0.12629
2,Sweden 2 x 0 Tunisia,2,0,9598,0.09598
3,Sweden 2 x 1 Tunisia,2,1,9246,0.09246
4,Sweden 0 x 0 Tunisia,0,0,9183,0.09183
5,Sweden 0 x 1 Tunisia,0,1,8277,0.08277
6,Sweden 1 x 2 Tunisia,1,2,5870,0.05870
7,Sweden 2 x 2 Tunisia,2,2,4522,0.04522
8,Sweden 3 x 0 Tunisia,3,0,4416,0.04416
9,Sweden 3 x 1 Tunisia,3,1,4311,0.04311


In [47]:
match_analysis = analyze_specific_match(
    home_team="Spain",
    away_team="Cabo Verde",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Spain,Cabo Verde,4.671952,0.462919,0.4,100000,2026-06-15


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Spain vence,0.963924,0.883753,0.931855,0.93312
1,Empate,0.027645,0.079538,0.048402,0.04744
2,Cabo Verde vence,0.008432,0.036710,0.019743,0.01944


,score,home_goals,away_goals,probability
0,Spain 4 x 0 Cabo Verde,4,0,0.113977
1,Spain 5 x 0 Cabo Verde,5,0,0.106499
2,Spain 3 x 0 Cabo Verde,3,0,0.097584
3,Spain 6 x 0 Cabo Verde,6,0,0.082927
4,Spain 2 x 0 Cabo Verde,2,0,0.062662
5,Spain 7 x 0 Cabo Verde,7,0,0.055347
6,Spain 4 x 1 Cabo Verde,4,1,0.052762
7,Spain 5 x 1 Cabo Verde,5,1,0.049301
8,Spain 3 x 1 Cabo Verde,3,1,0.045174
9,Spain 6 x 1 Cabo Verde,6,1,0.038388


,score,home_goals,away_goals,count,simulated_probability
0,Spain 4 x 0 Cabo Verde,4,0,11354,0.11354
1,Spain 5 x 0 Cabo Verde,5,0,10751,0.10751
2,Spain 3 x 0 Cabo Verde,3,0,9811,0.09811
3,Spain 6 x 0 Cabo Verde,6,0,8283,0.08283
4,Spain 2 x 0 Cabo Verde,2,0,6250,0.06250
5,Spain 7 x 0 Cabo Verde,7,0,5510,0.05510
6,Spain 4 x 1 Cabo Verde,4,1,5219,0.05219
7,Spain 5 x 1 Cabo Verde,5,1,5036,0.05036
8,Spain 3 x 1 Cabo Verde,3,1,4422,0.04422
9,Spain 6 x 1 Cabo Verde,6,1,3943,0.03943


In [48]:
match_analysis = analyze_specific_match(
    home_team="Belgium",
    away_team="Egypt",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Belgium,Egypt,1.878503,0.808913,0.4,100000,2026-06-15


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Belgium vence,0.626515,0.578870,0.607457,0.60963
1,Empate,0.218062,0.244906,0.228799,0.22621
2,Egypt vence,0.155423,0.176224,0.163744,0.16416


,score,home_goals,away_goals,probability
0,Belgium 1 x 0 Egypt,1,0,0.123956
1,Belgium 2 x 0 Egypt,2,0,0.116426
2,Belgium 1 x 1 Egypt,1,1,0.108508
3,Belgium 2 x 1 Egypt,2,1,0.094178
4,Belgium 3 x 0 Egypt,3,0,0.072902
5,Belgium 0 x 0 Egypt,0,0,0.071408
6,Belgium 3 x 1 Egypt,3,1,0.058971
7,Belgium 0 x 1 Egypt,0,1,0.057999
8,Belgium 1 x 2 Egypt,1,2,0.044066
9,Belgium 2 x 2 Egypt,2,2,0.041221


,score,home_goals,away_goals,count,simulated_probability
0,Belgium 1 x 0 Egypt,1,0,12428,0.12428
1,Belgium 2 x 0 Egypt,2,0,11690,0.11690
2,Belgium 1 x 1 Egypt,1,1,10634,0.10634
3,Belgium 2 x 1 Egypt,2,1,9544,0.09544
4,Belgium 3 x 0 Egypt,3,0,7339,0.07339
5,Belgium 0 x 0 Egypt,0,0,7130,0.07130
6,Belgium 3 x 1 Egypt,3,1,5872,0.05872
7,Belgium 0 x 1 Egypt,0,1,5799,0.05799
8,Belgium 1 x 2 Egypt,1,2,4434,0.04434
9,Belgium 2 x 2 Egypt,2,2,4088,0.04088


In [49]:
match_analysis = analyze_specific_match(
    home_team="Saudi Arabia",
    away_team="Uruguay",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Saudi Arabia,Uruguay,0.633539,1.475035,0.4,100000,2026-06-15


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Saudi Arabia vence,0.159392,0.122800,0.144755,0.14472
1,Empate,0.264294,0.216296,0.245095,0.24592
2,Uruguay vence,0.576314,0.660904,0.610150,0.60936


,score,home_goals,away_goals,probability
0,Saudi Arabia 0 x 1 Uruguay,0,1,0.189600
1,Saudi Arabia 0 x 2 Uruguay,0,2,0.139833
2,Saudi Arabia 0 x 0 Uruguay,0,0,0.112591
3,Saudi Arabia 1 x 1 Uruguay,1,1,0.105216
4,Saudi Arabia 1 x 2 Uruguay,1,2,0.088590
5,Saudi Arabia 1 x 0 Uruguay,1,0,0.069855
6,Saudi Arabia 0 x 3 Uruguay,0,3,0.068753
7,Saudi Arabia 1 x 3 Uruguay,1,3,0.043558
8,Saudi Arabia 2 x 1 Uruguay,2,1,0.032640
9,Saudi Arabia 0 x 4 Uruguay,0,4,0.025353


,score,home_goals,away_goals,count,simulated_probability
0,Saudi Arabia 0 x 1 Uruguay,0,1,18947,0.18947
1,Saudi Arabia 0 x 2 Uruguay,0,2,13896,0.13896
2,Saudi Arabia 0 x 0 Uruguay,0,0,11216,0.11216
3,Saudi Arabia 1 x 1 Uruguay,1,1,10680,0.10680
4,Saudi Arabia 1 x 2 Uruguay,1,2,8934,0.08934
5,Saudi Arabia 1 x 0 Uruguay,1,0,7012,0.07012
6,Saudi Arabia 0 x 3 Uruguay,0,3,6861,0.06861
7,Saudi Arabia 1 x 3 Uruguay,1,3,4281,0.04281
8,Saudi Arabia 2 x 1 Uruguay,2,1,3259,0.03259
9,Saudi Arabia 0 x 4 Uruguay,0,4,2541,0.02541


In [50]:
match_analysis = analyze_specific_match(
    home_team="Iran",
    away_team="New Zealand",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Iran,New Zealand,1.65981,0.995607,0.4,100000,2026-06-15


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Iran vence,0.529283,0.513520,0.522978,0.52492
1,Empate,0.244146,0.275366,0.256634,0.25578
2,New Zealand vence,0.226571,0.211114,0.220388,0.21930


,score,home_goals,away_goals,probability
0,Iran 1 x 1 New Zealand,1,1,0.122061
1,Iran 1 x 0 New Zealand,1,0,0.115245
2,Iran 2 x 0 New Zealand,2,0,0.095642
3,Iran 2 x 1 New Zealand,2,1,0.095222
4,Iran 0 x 0 New Zealand,0,0,0.073864
5,Iran 0 x 1 New Zealand,0,1,0.068052
6,Iran 1 x 2 New Zealand,1,2,0.056228
7,Iran 3 x 0 New Zealand,3,0,0.052916
8,Iran 3 x 1 New Zealand,3,1,0.052684
9,Iran 2 x 2 New Zealand,2,2,0.050427


,score,home_goals,away_goals,count,simulated_probability
0,Iran 1 x 1 New Zealand,1,1,12111,0.12111
1,Iran 1 x 0 New Zealand,1,0,11527,0.11527
2,Iran 2 x 1 New Zealand,2,1,9657,0.09657
3,Iran 2 x 0 New Zealand,2,0,9609,0.09609
4,Iran 0 x 0 New Zealand,0,0,7337,0.07337
5,Iran 0 x 1 New Zealand,0,1,6777,0.06777
6,Iran 1 x 2 New Zealand,1,2,5502,0.05502
7,Iran 3 x 0 New Zealand,3,0,5337,0.05337
8,Iran 3 x 1 New Zealand,3,1,5228,0.05228
9,Iran 2 x 2 New Zealand,2,2,5113,0.05113


In [51]:
match_analysis = analyze_specific_match(
    home_team="France",
    away_team="Senegal",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,France,Senegal,2.076625,0.865824,0.4,100000,2026-06-16


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,France vence,0.653762,0.650304,0.652379,0.65175
1,Empate,0.199724,0.215783,0.206147,0.20829
2,Senegal vence,0.146514,0.133913,0.141474,0.13996


,score,home_goals,away_goals,probability
0,France 2 x 0 Senegal,2,0,0.113470
1,France 1 x 0 Senegal,1,0,0.109283
2,France 2 x 1 Senegal,2,1,0.098245
3,France 1 x 1 Senegal,1,1,0.097870
4,France 3 x 0 Senegal,3,0,0.078545
5,France 3 x 1 Senegal,3,1,0.068006
6,France 0 x 0 Senegal,0,0,0.054433
7,France 0 x 1 Senegal,0,1,0.044090
8,France 2 x 2 Senegal,2,2,0.043993
9,France 4 x 0 Senegal,4,0,0.040777


,score,home_goals,away_goals,count,simulated_probability
0,France 2 x 0 Senegal,2,0,11259,0.11259
1,France 1 x 0 Senegal,1,0,10886,0.10886
2,France 2 x 1 Senegal,2,1,9892,0.09892
3,France 1 x 1 Senegal,1,1,9782,0.09782
4,France 3 x 0 Senegal,3,0,7872,0.07872
5,France 3 x 1 Senegal,3,1,6881,0.06881
6,France 0 x 0 Senegal,0,0,5488,0.05488
7,France 2 x 2 Senegal,2,2,4564,0.04564
8,France 0 x 1 Senegal,0,1,4295,0.04295
9,France 4 x 0 Senegal,4,0,4027,0.04027


In [52]:
match_analysis = analyze_specific_match(
    home_team="Iraq",
    away_team="Norway",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Iraq,Norway,0.709396,2.032108,0.4,100000,2026-06-16


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Iraq vence,0.119690,0.068182,0.099087,0.09933
1,Empate,0.196791,0.136364,0.172620,0.17318
2,Norway vence,0.683519,0.795455,0.728293,0.72749


,score,home_goals,away_goals,probability
0,Iraq 0 x 2 Norway,0,2,0.141842
1,Iraq 0 x 1 Norway,0,1,0.139600
2,Iraq 1 x 2 Norway,1,2,0.100622
3,Iraq 0 x 3 Norway,0,3,0.096079
4,Iraq 1 x 1 Norway,1,1,0.081528
5,Iraq 1 x 3 Norway,1,3,0.068158
6,Iraq 0 x 0 Norway,0,0,0.056555
7,Iraq 0 x 4 Norway,0,4,0.048811
8,Iraq 1 x 0 Norway,1,0,0.037864
9,Iraq 1 x 4 Norway,1,4,0.034626


,score,home_goals,away_goals,count,simulated_probability
0,Iraq 0 x 2 Norway,0,2,14147,0.14147
1,Iraq 0 x 1 Norway,0,1,13844,0.13844
2,Iraq 1 x 2 Norway,1,2,10159,0.10159
3,Iraq 0 x 3 Norway,0,3,9545,0.09545
4,Iraq 1 x 1 Norway,1,1,8221,0.08221
5,Iraq 1 x 3 Norway,1,3,6884,0.06884
6,Iraq 0 x 0 Norway,0,0,5699,0.05699
7,Iraq 0 x 4 Norway,0,4,4801,0.04801
8,Iraq 1 x 0 Norway,1,0,3805,0.03805
9,Iraq 1 x 4 Norway,1,4,3387,0.03387


In [53]:
match_analysis = analyze_specific_match(
    home_team="Argentina",
    away_team="Algeria",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Argentina,Algeria,2.121526,0.686068,0.4,100000,2026-06-16


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Argentina vence,0.706431,0.686522,0.698467,0.69806
1,Empate,0.185824,0.207449,0.194474,0.19507
2,Algeria vence,0.107745,0.106029,0.107059,0.10687


,score,home_goals,away_goals,probability
0,Argentina 2 x 0 Algeria,2,0,0.134285
1,Argentina 1 x 0 Algeria,1,0,0.126593
2,Argentina 3 x 0 Algeria,3,0,0.094963
3,Argentina 2 x 1 Algeria,2,1,0.092128
4,Argentina 1 x 1 Algeria,1,1,0.091930
5,Argentina 3 x 1 Algeria,3,1,0.065151
6,Argentina 0 x 0 Algeria,0,0,0.063160
7,Argentina 4 x 0 Algeria,4,0,0.050367
8,Argentina 0 x 1 Algeria,0,1,0.041141
9,Argentina 4 x 1 Algeria,4,1,0.034555


,score,home_goals,away_goals,count,simulated_probability
0,Argentina 2 x 0 Algeria,2,0,13393,0.13393
1,Argentina 1 x 0 Algeria,1,0,12663,0.12663
2,Argentina 3 x 0 Algeria,3,0,9539,0.09539
3,Argentina 2 x 1 Algeria,2,1,9198,0.09198
4,Argentina 1 x 1 Algeria,1,1,9119,0.09119
5,Argentina 3 x 1 Algeria,3,1,6578,0.06578
6,Argentina 0 x 0 Algeria,0,0,6316,0.06316
7,Argentina 4 x 0 Algeria,4,0,5011,0.05011
8,Argentina 0 x 1 Algeria,0,1,4060,0.04060
9,Argentina 2 x 2 Algeria,2,2,3469,0.03469


In [54]:
match_analysis = analyze_specific_match(
    home_team="Austria",
    away_team="Jordan",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Austria,Jordan,1.656249,0.918329,0.4,100000,2026-06-16


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Austria vence,0.547474,0.726704,0.619166,0.62102
1,Empate,0.244325,0.173088,0.215830,0.21385
2,Jordan vence,0.208201,0.100209,0.165004,0.16513


,score,home_goals,away_goals,probability
0,Austria 1 x 0 Jordan,1,0,0.142707
1,Austria 2 x 0 Jordan,2,0,0.118179
2,Austria 2 x 1 Jordan,2,1,0.108527
3,Austria 1 x 1 Jordan,1,1,0.102363
4,Austria 0 x 0 Jordan,0,0,0.067301
5,Austria 3 x 0 Jordan,3,0,0.065245
6,Austria 3 x 1 Jordan,3,1,0.059916
7,Austria 0 x 1 Jordan,0,1,0.055448
8,Austria 1 x 2 Jordan,1,2,0.042168
9,Austria 2 x 2 Jordan,2,2,0.038923


,score,home_goals,away_goals,count,simulated_probability
0,Austria 1 x 0 Jordan,1,0,14234,0.14234
1,Austria 2 x 0 Jordan,2,0,11852,0.11852
2,Austria 2 x 1 Jordan,2,1,11028,0.11028
3,Austria 1 x 1 Jordan,1,1,10081,0.10081
4,Austria 0 x 0 Jordan,0,0,6700,0.06700
5,Austria 3 x 0 Jordan,3,0,6592,0.06592
6,Austria 3 x 1 Jordan,3,1,5917,0.05917
7,Austria 0 x 1 Jordan,0,1,5515,0.05515
8,Austria 1 x 2 Jordan,1,2,4249,0.04249
9,Austria 2 x 2 Jordan,2,2,3926,0.03926


In [55]:
match_analysis = analyze_specific_match(
    home_team="Portugal",
    away_team="Congo DR",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Portugal,Congo DR,2.541204,0.643164,0.4,100000,2026-06-17


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Portugal vence,0.785016,0.746841,0.769746,0.76925
1,Empate,0.142307,0.166253,0.151885,0.15340
2,Congo DR vence,0.072677,0.086905,0.078369,0.07735


,score,home_goals,away_goals,probability
0,Portugal 2 x 0 Congo DR,2,0,0.131098
1,Portugal 3 x 0 Congo DR,3,0,0.111049
2,Portugal 1 x 0 Congo DR,1,0,0.103178
3,Portugal 2 x 1 Congo DR,2,1,0.084317
4,Portugal 1 x 1 Congo DR,1,1,0.072232
5,Portugal 3 x 1 Congo DR,3,1,0.071423
6,Portugal 4 x 0 Congo DR,4,0,0.070549
7,Portugal 4 x 1 Congo DR,4,1,0.045375
8,Portugal 0 x 0 Congo DR,0,0,0.044194
9,Portugal 5 x 0 Congo DR,5,0,0.035856


,score,home_goals,away_goals,count,simulated_probability
0,Portugal 2 x 0 Congo DR,2,0,12931,0.12931
1,Portugal 3 x 0 Congo DR,3,0,11245,0.11245
2,Portugal 1 x 0 Congo DR,1,0,10292,0.10292
3,Portugal 2 x 1 Congo DR,2,1,8396,0.08396
4,Portugal 1 x 1 Congo DR,1,1,7228,0.07228
5,Portugal 3 x 1 Congo DR,3,1,7213,0.07213
6,Portugal 4 x 0 Congo DR,4,0,7103,0.07103
7,Portugal 0 x 0 Congo DR,0,0,4506,0.04506
8,Portugal 4 x 1 Congo DR,4,1,4491,0.04491
9,Portugal 5 x 0 Congo DR,5,0,3547,0.03547


In [56]:
match_analysis = analyze_specific_match(
    home_team="England",
    away_team="Croatia",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,England,Croatia,1.823166,0.930868,0.4,100000,2026-06-17


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,England vence,0.583698,0.553989,0.571814,0.57364
1,Empate,0.227202,0.250753,0.236623,0.23423
2,Croatia vence,0.189100,0.195258,0.191563,0.19213


,score,home_goals,away_goals,probability
0,England 1 x 0 Croatia,1,0,0.113719
1,England 1 x 1 Croatia,1,1,0.112538
2,England 2 x 0 Croatia,2,0,0.103664
3,England 2 x 1 Croatia,2,1,0.096498
4,England 0 x 0 Croatia,0,0,0.066311
5,England 3 x 0 Croatia,3,0,0.062999
6,England 0 x 1 Croatia,0,1,0.060041
7,England 3 x 1 Croatia,3,1,0.058644
8,England 1 x 2 Croatia,1,2,0.050949
9,England 2 x 2 Croatia,2,2,0.047748


,score,home_goals,away_goals,count,simulated_probability
0,England 1 x 0 Croatia,1,0,11398,0.11398
1,England 1 x 1 Croatia,1,1,11038,0.11038
2,England 2 x 0 Croatia,2,0,10440,0.10440
3,England 2 x 1 Croatia,2,1,9783,0.09783
4,England 0 x 0 Croatia,0,0,6622,0.06622
5,England 3 x 0 Croatia,3,0,6307,0.06307
6,England 0 x 1 Croatia,0,1,5966,0.05966
7,England 3 x 1 Croatia,3,1,5837,0.05837
8,England 1 x 2 Croatia,1,2,5175,0.05175
9,England 2 x 2 Croatia,2,2,4741,0.04741


In [57]:
match_analysis = analyze_specific_match(
    home_team="Ghana",
    away_team="Panama",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Ghana,Panama,1.062957,1.417115,0.4,100000,2026-06-17


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Ghana vence,0.282709,0.459770,0.353534,0.35280
1,Empate,0.266120,0.275862,0.270017,0.26963
2,Panama vence,0.451171,0.264368,0.376450,0.37757


,score,home_goals,away_goals,probability
0,Ghana 1 x 1 Panama,1,1,0.127983
1,Ghana 1 x 0 Panama,1,0,0.111308
2,Ghana 0 x 1 Panama,0,1,0.099012
3,Ghana 0 x 0 Panama,0,0,0.084963
4,Ghana 2 x 1 Panama,2,1,0.083833
5,Ghana 1 x 2 Panama,1,2,0.074573
6,Ghana 0 x 2 Panama,0,2,0.070156
7,Ghana 2 x 0 Panama,2,0,0.059158
8,Ghana 2 x 2 Panama,2,2,0.048196
9,Ghana 1 x 3 Panama,1,3,0.035226


,score,home_goals,away_goals,count,simulated_probability
0,Ghana 1 x 1 Panama,1,1,12842,0.12842
1,Ghana 1 x 0 Panama,1,0,10978,0.10978
2,Ghana 0 x 1 Panama,0,1,9881,0.09881
3,Ghana 0 x 0 Panama,0,0,8477,0.08477
4,Ghana 2 x 1 Panama,2,1,8458,0.08458
5,Ghana 1 x 2 Panama,1,2,7493,0.07493
6,Ghana 0 x 2 Panama,0,2,7001,0.07001
7,Ghana 2 x 0 Panama,2,0,5966,0.05966
8,Ghana 2 x 2 Panama,2,2,4698,0.04698
9,Ghana 1 x 3 Panama,1,3,3638,0.03638


In [58]:
match_analysis = analyze_specific_match(
    home_team="Uzbekistan",
    away_team="Colombia",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Uzbekistan,Colombia,0.849559,1.936324,0.4,100000,2026-06-17


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Uzbekistan vence,0.157886,0.106029,0.137144,0.13487
1,Empate,0.213321,0.207449,0.210972,0.21254
2,Colombia vence,0.628793,0.686522,0.651884,0.65259


,score,home_goals,away_goals,probability
0,Uzbekistan 0 x 1 Colombia,0,1,0.123808
1,Uzbekistan 0 x 2 Colombia,0,2,0.119867
2,Uzbekistan 1 x 2 Colombia,1,2,0.101834
3,Uzbekistan 1 x 1 Colombia,1,1,0.100340
4,Uzbekistan 0 x 3 Colombia,0,3,0.077367
5,Uzbekistan 1 x 3 Colombia,1,3,0.065728
6,Uzbekistan 0 x 0 Colombia,0,0,0.060996
7,Uzbekistan 1 x 0 Colombia,1,0,0.045513
8,Uzbekistan 2 x 2 Colombia,2,2,0.041265
9,Uzbekistan 0 x 4 Colombia,0,4,0.037452


,score,home_goals,away_goals,count,simulated_probability
0,Uzbekistan 0 x 1 Colombia,0,1,12315,0.12315
1,Uzbekistan 0 x 2 Colombia,0,2,11970,0.11970
2,Uzbekistan 1 x 2 Colombia,1,2,10380,0.10380
3,Uzbekistan 1 x 1 Colombia,1,1,10149,0.10149
4,Uzbekistan 0 x 3 Colombia,0,3,7668,0.07668
5,Uzbekistan 1 x 3 Colombia,1,3,6561,0.06561
6,Uzbekistan 0 x 0 Colombia,0,0,6123,0.06123
7,Uzbekistan 1 x 0 Colombia,1,0,4406,0.04406
8,Uzbekistan 2 x 2 Colombia,2,2,4117,0.04117
9,Uzbekistan 0 x 4 Colombia,0,4,3680,0.03680


In [60]:
match_analysis = analyze_specific_match(
    home_team="Czechia",
    away_team="South Africa",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Czechia,South Africa,1.516391,0.760148,0.4,100000,2026-06-18


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Czechia vence,0.552900,0.483906,0.525302,0.52556
1,Empate,0.259738,0.285944,0.270221,0.26913
2,South Africa vence,0.187361,0.230150,0.204477,0.20531


,score,home_goals,away_goals,probability
0,Czechia 1 x 0 South Africa,1,0,0.147872
1,Czechia 1 x 1 South Africa,1,1,0.123085
2,Czechia 2 x 0 South Africa,2,0,0.112116
3,Czechia 0 x 0 South Africa,0,0,0.106781
4,Czechia 2 x 1 South Africa,2,1,0.085225
5,Czechia 0 x 1 South Africa,0,1,0.085148
6,Czechia 3 x 0 South Africa,3,0,0.056670
7,Czechia 1 x 2 South Africa,1,2,0.049074
8,Czechia 3 x 1 South Africa,3,1,0.043078
9,Czechia 2 x 2 South Africa,2,2,0.035469


,score,home_goals,away_goals,count,simulated_probability
0,Czechia 1 x 0 South Africa,1,0,14755,0.14755
1,Czechia 1 x 1 South Africa,1,1,12212,0.12212
2,Czechia 2 x 0 South Africa,2,0,11326,0.11326
3,Czechia 0 x 0 South Africa,0,0,10611,0.10611
4,Czechia 2 x 1 South Africa,2,1,8522,0.08522
5,Czechia 0 x 1 South Africa,0,1,8503,0.08503
6,Czechia 3 x 0 South Africa,3,0,5578,0.05578
7,Czechia 1 x 2 South Africa,1,2,4986,0.04986
8,Czechia 3 x 1 South Africa,3,1,4264,0.04264
9,Czechia 2 x 2 South Africa,2,2,3636,0.03636


In [61]:
match_analysis = analyze_specific_match(
    home_team="Switzerland",
    away_team="Bosnia and Herzegovina",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Switzerland,Bosnia and Herzegovina,2.69542,0.639166,0.4,100000,2026-06-18


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Switzerland vence,0.806769,0.589632,0.719915,0.72021
1,Empate,0.129111,0.238838,0.173002,0.17526
2,Bosnia and Herzegovina vence,0.064120,0.171529,0.107084,0.10453


,score,home_goals,away_goals,probability
0,Switzerland 2 x 0 Bosnia and Herzegovina,2,0,0.115508
1,Switzerland 3 x 0 Bosnia and Herzegovina,3,0,0.103781
2,Switzerland 1 x 0 Bosnia and Herzegovina,1,0,0.085707
3,Switzerland 1 x 1 Bosnia and Herzegovina,1,1,0.082260
4,Switzerland 2 x 1 Bosnia and Herzegovina,2,1,0.073829
5,Switzerland 4 x 0 Bosnia and Herzegovina,4,0,0.069933
6,Switzerland 3 x 1 Bosnia and Herzegovina,3,1,0.066333
7,Switzerland 0 x 0 Bosnia and Herzegovina,0,0,0.047747
8,Switzerland 4 x 1 Bosnia and Herzegovina,4,1,0.044699
9,Switzerland 0 x 1 Bosnia and Herzegovina,0,1,0.038037


,score,home_goals,away_goals,count,simulated_probability
0,Switzerland 2 x 0 Bosnia and Herzegovina,2,0,11454,0.11454
1,Switzerland 3 x 0 Bosnia and Herzegovina,3,0,10524,0.10524
2,Switzerland 1 x 0 Bosnia and Herzegovina,1,0,8598,0.08598
3,Switzerland 1 x 1 Bosnia and Herzegovina,1,1,8289,0.08289
4,Switzerland 2 x 1 Bosnia and Herzegovina,2,1,7294,0.07294
5,Switzerland 4 x 0 Bosnia and Herzegovina,4,0,7058,0.07058
6,Switzerland 3 x 1 Bosnia and Herzegovina,3,1,6643,0.06643
7,Switzerland 0 x 0 Bosnia and Herzegovina,0,0,4839,0.04839
8,Switzerland 4 x 1 Bosnia and Herzegovina,4,1,4356,0.04356
9,Switzerland 0 x 1 Bosnia and Herzegovina,0,1,3720,0.03720


In [62]:
match_analysis = analyze_specific_match(
    home_team="Canada",
    away_team="Qatar",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Canada,Qatar,2.068993,0.528122,0.4,100000,2026-06-18


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Canada vence,0.736423,0.722073,0.730683,0.73199
1,Empate,0.181008,0.184058,0.182228,0.18168
2,Qatar vence,0.082569,0.093869,0.087089,0.08633


,score,home_goals,away_goals,probability
0,Canada 2 x 0 Qatar,2,0,0.158191
1,Canada 1 x 0 Qatar,1,0,0.152916
2,Canada 3 x 0 Qatar,3,0,0.109099
3,Canada 2 x 1 Qatar,2,1,0.083544
4,Canada 1 x 1 Qatar,1,1,0.081941
5,Canada 0 x 0 Qatar,0,0,0.074991
6,Canada 3 x 1 Qatar,3,1,0.057617
7,Canada 4 x 0 Qatar,4,0,0.056431
8,Canada 0 x 1 Qatar,0,1,0.041493
9,Canada 4 x 1 Qatar,4,1,0.029803


,score,home_goals,away_goals,count,simulated_probability
0,Canada 2 x 0 Qatar,2,0,15895,0.15895
1,Canada 1 x 0 Qatar,1,0,15290,0.15290
2,Canada 3 x 0 Qatar,3,0,10891,0.10891
3,Canada 2 x 1 Qatar,2,1,8439,0.08439
4,Canada 1 x 1 Qatar,1,1,8122,0.08122
5,Canada 0 x 0 Qatar,0,0,7454,0.07454
6,Canada 3 x 1 Qatar,3,1,5781,0.05781
7,Canada 4 x 0 Qatar,4,0,5603,0.05603
8,Canada 0 x 1 Qatar,0,1,4153,0.04153
9,Canada 4 x 1 Qatar,4,1,2998,0.02998


In [63]:
match_analysis = analyze_specific_match(
    home_team="Mexico",
    away_team="South Korea",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Mexico,South Korea,1.063029,0.562244,0.4,100000,2026-06-18


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Mexico vence,0.473881,0.525006,0.494331,0.49431
1,Empate,0.333309,0.270003,0.307987,0.30902
2,South Korea vence,0.192810,0.204991,0.197683,0.19667


,score,home_goals,away_goals,probability
0,Mexico 1 x 0 South Korea,1,0,0.218296
1,Mexico 0 x 0 South Korea,0,0,0.181902
2,Mexico 2 x 0 South Korea,2,0,0.116028
3,Mexico 0 x 1 South Korea,0,1,0.113479
4,Mexico 1 x 1 South Korea,1,1,0.108720
5,Mexico 2 x 1 South Korea,2,1,0.065236
6,Mexico 3 x 0 South Korea,3,0,0.041114
7,Mexico 1 x 2 South Korea,1,2,0.033912
8,Mexico 0 x 2 South Korea,0,2,0.031902
9,Mexico 3 x 1 South Korea,3,1,0.023116


,score,home_goals,away_goals,count,simulated_probability
0,Mexico 1 x 0 South Korea,1,0,21791,0.21791
1,Mexico 0 x 0 South Korea,0,0,18151,0.18151
2,Mexico 2 x 0 South Korea,2,0,11703,0.11703
3,Mexico 0 x 1 South Korea,0,1,11349,0.11349
4,Mexico 1 x 1 South Korea,1,1,11039,0.11039
5,Mexico 2 x 1 South Korea,2,1,6424,0.06424
6,Mexico 3 x 0 South Korea,3,0,4095,0.04095
7,Mexico 1 x 2 South Korea,1,2,3373,0.03373
8,Mexico 0 x 2 South Korea,0,2,3140,0.03140
9,Mexico 3 x 1 South Korea,3,1,2377,0.02377


In [64]:
match_analysis = analyze_specific_match(
    home_team="United States",
    away_team="Australia",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,United States,Australia,1.606717,0.932553,0.4,100000,2026-06-19


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,United States vence,0.531696,0.542134,0.535871,0.53762
1,Empate,0.249586,0.248240,0.249047,0.24786
2,Australia vence,0.218719,0.209625,0.215081,0.21452


,score,home_goals,away_goals,probability
0,United States 1 x 0 Australia,1,0,0.127804
1,United States 1 x 1 Australia,1,1,0.118001
2,United States 2 x 0 Australia,2,0,0.102673
3,United States 2 x 1 Australia,2,1,0.095748
4,United States 0 x 0 Australia,0,0,0.078754
5,United States 0 x 1 Australia,0,1,0.072377
6,United States 3 x 0 Australia,3,0,0.054989
7,United States 1 x 2 Australia,1,2,0.054223
8,United States 3 x 1 Australia,3,1,0.051280
9,United States 2 x 2 Australia,2,2,0.044202


,score,home_goals,away_goals,count,simulated_probability
0,United States 1 x 0 Australia,1,0,12777,0.12777
1,United States 1 x 1 Australia,1,1,11724,0.11724
2,United States 2 x 0 Australia,2,0,10337,0.10337
3,United States 2 x 1 Australia,2,1,9698,0.09698
4,United States 0 x 0 Australia,0,0,7851,0.07851
5,United States 0 x 1 Australia,0,1,7176,0.07176
6,United States 3 x 0 Australia,3,0,5476,0.05476
7,United States 1 x 2 Australia,1,2,5346,0.05346
8,United States 3 x 1 Australia,3,1,5098,0.05098
9,United States 2 x 2 Australia,2,2,4430,0.04430


In [65]:
match_analysis = analyze_specific_match(
    home_team="Morocco",
    away_team="Scotland",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Scotland,Morocco,1.164878,1.450174,0.4,100000,2026-06-19


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Scotland vence,0.303629,0.231444,0.274755,0.27371
1,Empate,0.259859,0.289305,0.271637,0.27124
2,Morocco vence,0.436512,0.479252,0.453608,0.45505


,score,home_goals,away_goals,probability
0,Scotland 1 x 1 Morocco,1,1,0.129196
1,Scotland 0 x 1 Morocco,0,1,0.110256
2,Scotland 1 x 2 Morocco,1,2,0.093126
3,Scotland 0 x 2 Morocco,0,2,0.079945
4,Scotland 1 x 0 Morocco,1,0,0.077122
5,Scotland 0 x 0 Morocco,0,0,0.076480
6,Scotland 2 x 1 Morocco,2,1,0.065140
7,Scotland 2 x 2 Morocco,2,2,0.054562
8,Scotland 1 x 3 Morocco,1,3,0.045016
9,Scotland 2 x 0 Morocco,2,0,0.044919


,score,home_goals,away_goals,count,simulated_probability
0,Scotland 1 x 1 Morocco,1,1,12980,0.12980
1,Scotland 0 x 1 Morocco,0,1,11001,0.11001
2,Scotland 1 x 2 Morocco,1,2,9373,0.09373
3,Scotland 0 x 2 Morocco,0,2,8057,0.08057
4,Scotland 0 x 0 Morocco,0,0,7607,0.07607
5,Scotland 1 x 0 Morocco,1,0,7588,0.07588
6,Scotland 2 x 1 Morocco,2,1,6587,0.06587
7,Scotland 2 x 2 Morocco,2,2,5335,0.05335
8,Scotland 1 x 3 Morocco,1,3,4606,0.04606
9,Scotland 2 x 0 Morocco,2,0,4516,0.04516


In [66]:
match_analysis = analyze_specific_match(
    home_team="Brazil",
    away_team="Haiti",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Brazil,Haiti,2.817461,0.571718,0.4,100000,2026-06-19


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Brazil vence,0.835155,0.884053,0.854714,0.85459
1,Empate,0.114319,0.083024,0.101801,0.10127
2,Haiti vence,0.050526,0.032923,0.043485,0.04414


,score,home_goals,away_goals,probability
0,Brazil 2 x 0 Haiti,2,0,0.137061
1,Brazil 3 x 0 Haiti,3,0,0.128721
2,Brazil 1 x 0 Haiti,1,0,0.097294
3,Brazil 4 x 0 Haiti,4,0,0.090667
4,Brazil 2 x 1 Haiti,2,1,0.078360
5,Brazil 3 x 1 Haiti,3,1,0.073592
6,Brazil 4 x 1 Haiti,4,1,0.051836
7,Brazil 5 x 0 Haiti,5,0,0.051090
8,Brazil 1 x 1 Haiti,1,1,0.048400
9,Brazil 0 x 0 Haiti,0,0,0.030047


,score,home_goals,away_goals,count,simulated_probability
0,Brazil 2 x 0 Haiti,2,0,13667,0.13667
1,Brazil 3 x 0 Haiti,3,0,12846,0.12846
2,Brazil 1 x 0 Haiti,1,0,9598,0.09598
3,Brazil 4 x 0 Haiti,4,0,9069,0.09069
4,Brazil 2 x 1 Haiti,2,1,7809,0.07809
5,Brazil 3 x 1 Haiti,3,1,7549,0.07549
6,Brazil 4 x 1 Haiti,4,1,5244,0.05244
7,Brazil 5 x 0 Haiti,5,0,5061,0.05061
8,Brazil 1 x 1 Haiti,1,1,4832,0.04832
9,Brazil 0 x 0 Haiti,0,0,3018,0.03018


In [67]:
match_analysis = analyze_specific_match(
    home_team="Turkey",
    away_team="Paraguay",
    schedule_data=schedule_data,
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    market_weight=0.4,
    max_goals=10,
    simulations=100_000,
    seed=42,
)

display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,market_weight,simulations,match_date
0,Turkey,Paraguay,1.265255,1.164009,0.4,100000,2026-06-19


,outcome,model_probability,market_probability,final_probability,simulated_probability
0,Turkey vence,0.387313,0.426402,0.402949,0.40307
1,Empate,0.274211,0.299835,0.284461,0.28354
2,Paraguay vence,0.338476,0.273763,0.312591,0.31339


,score,home_goals,away_goals,probability
0,Turkey 1 x 1 Paraguay,1,1,0.134603
1,Turkey 1 x 0 Paraguay,1,0,0.115971
2,Turkey 0 x 1 Paraguay,0,1,0.094708
3,Turkey 0 x 0 Paraguay,0,0,0.091395
4,Turkey 2 x 1 Paraguay,2,1,0.085399
5,Turkey 2 x 0 Paraguay,2,0,0.073367
6,Turkey 1 x 2 Paraguay,1,2,0.069742
7,Turkey 0 x 2 Paraguay,0,2,0.055121
8,Turkey 2 x 2 Paraguay,2,2,0.049560
9,Turkey 3 x 1 Paraguay,3,1,0.036017


,score,home_goals,away_goals,count,simulated_probability
0,Turkey 1 x 1 Paraguay,1,1,13455,0.13455
1,Turkey 1 x 0 Paraguay,1,0,11425,0.11425
2,Turkey 0 x 1 Paraguay,0,1,9452,0.09452
3,Turkey 0 x 0 Paraguay,0,0,9106,0.09106
4,Turkey 2 x 1 Paraguay,2,1,8602,0.08602
5,Turkey 2 x 0 Paraguay,2,0,7409,0.07409
6,Turkey 1 x 2 Paraguay,1,2,6987,0.06987
7,Turkey 0 x 2 Paraguay,0,2,5481,0.05481
8,Turkey 2 x 2 Paraguay,2,2,4933,0.04933
9,Turkey 3 x 1 Paraguay,3,1,3588,0.03588


In [46]:
def analyze_hypothetical_match(
    home_team,
    away_team,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    match_date="2026-07-01",
    neutral=1,
    tournament="FIFA World Cup",
    stage="knockout",
    market_probabilities=None,
    market_weight=0.35,
    max_goals=10,
    simulations=100_000,
    seed=42,
):
    match_date = pd.Timestamp(match_date)

    if market_probabilities is None:
        market_probabilities = path_a.DEFAULT_MARKET_PROBABILITIES.copy()

    feature_row = path_a.build_feature_row(
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        neutral=neutral,
        tournament=tournament,
        states=initial_states,
        ranking_lookup=ranking_lookup,
        market_probabilities=market_probabilities,
    )

    home_lambda, away_lambda = path_a.predict_expected_goals(
        home_model,
        away_model,
        feature_row,
    )

    original_score_matrix = calculate_score_probability_matrix(
        home_lambda=home_lambda,
        away_lambda=away_lambda,
        max_goals=max_goals,
    )

    model_probabilities = calculate_outcome_probabilities_from_score_matrix(
        original_score_matrix
    )

    target_probabilities = blend_outcome_probabilities(
        model_probabilities=model_probabilities,
        market_probabilities=market_probabilities,
        market_weight=market_weight,
    )

    adjusted_score_matrix = adjust_score_matrix_to_target_probabilities(
        score_matrix=original_score_matrix,
        target_probabilities=target_probabilities,
    )

    adjusted_probabilities = calculate_outcome_probabilities_from_score_matrix(
        adjusted_score_matrix
    )

    rng = np.random.default_rng(seed)

    flat_probabilities = adjusted_score_matrix.ravel()

    sampled_indices = rng.choice(
        len(flat_probabilities),
        size=simulations,
        replace=True,
        p=flat_probabilities,
    )

    sampled_scores = [
        np.unravel_index(index, adjusted_score_matrix.shape)
        for index in sampled_indices
    ]

    score_counter = Counter(sampled_scores)

    simulated_outcomes = {
        "home_win": 0,
        "draw": 0,
        "away_win": 0,
    }

    for (home_goals, away_goals), count in score_counter.items():
        outcome = get_score_outcome(home_goals, away_goals)
        simulated_outcomes[outcome] += count

    probabilities_summary = pd.DataFrame(
        [
            {
                "outcome": f"{home_team} vence",
                "model_probability": model_probabilities["home_win"],
                "final_probability": adjusted_probabilities["home_win"],
                "simulated_probability": simulated_outcomes["home_win"] / simulations,
            },
            {
                "outcome": "Empate",
                "model_probability": model_probabilities["draw"],
                "final_probability": adjusted_probabilities["draw"],
                "simulated_probability": simulated_outcomes["draw"] / simulations,
            },
            {
                "outcome": f"{away_team} vence",
                "model_probability": model_probabilities["away_win"],
                "final_probability": adjusted_probabilities["away_win"],
                "simulated_probability": simulated_outcomes["away_win"] / simulations,
            },
        ]
    )

    top_scorelines = get_top_scorelines(
        score_matrix=adjusted_score_matrix,
        home_team=home_team,
        away_team=away_team,
        top_n=20,
    )

    simulated_scorelines = pd.DataFrame(
        [
            {
                "score": f"{home_team} {home_goals} x {away_goals} {away_team}",
                "home_goals": home_goals,
                "away_goals": away_goals,
                "count": count,
                "simulated_probability": count / simulations,
            }
            for (home_goals, away_goals), count in score_counter.most_common(20)
        ]
    )

    lambdas_data = pd.DataFrame(
        [
            {
                "home_team": home_team,
                "away_team": away_team,
                "home_lambda": home_lambda,
                "away_lambda": away_lambda,
                "stage": stage,
                "market_weight": market_weight,
                "simulations": simulations,
            }
        ]
    )

    return {
        "lambdas": lambdas_data,
        "probabilities": probabilities_summary,
        "top_scorelines_exact": top_scorelines,
        "top_scorelines_simulated": simulated_scorelines,
        "score_matrix": adjusted_score_matrix,
    }

In [47]:
market_probabilities = {
    "market_home_prob": 0.33,
    "market_draw_prob": 0.34,
    "market_away_prob": 0.33,
}

In [54]:
match_analysis = analyze_hypothetical_match(
    home_team="Brazil",
    away_team="France",
    initial_states=initial_states,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
    home_model=home_model,
    away_model=away_model,
    match_date="2026-07-04",
    stage="final",
    market_probabilities=market_probabilities,
    market_weight=0,
    simulations=100_000,
    seed=42,
)

In [55]:
display(match_analysis["lambdas"])
display(match_analysis["probabilities"])
display(match_analysis["top_scorelines_exact"])
display(match_analysis["top_scorelines_simulated"])

,home_team,away_team,home_lambda,away_lambda,stage,market_weight,simulations
0,Brazil,France,1.209706,1.241664,final,0,100000


,outcome,model_probability,final_probability,simulated_probability
0,Brazil vence,0.355748,0.355748,0.35434
1,Empate,0.273143,0.273143,0.27327
2,France vence,0.371109,0.371109,0.37239


,score,home_goals,away_goals,probability
0,Brazil 1 x 1 France,1,1,0.129440
1,Brazil 0 x 1 France,0,1,0.107001
2,Brazil 1 x 0 France,1,0,0.104247
3,Brazil 0 x 0 France,0,0,0.086175
4,Brazil 1 x 2 France,1,2,0.080360
5,Brazil 2 x 1 France,2,1,0.078292
6,Brazil 0 x 2 France,0,2,0.066430
7,Brazil 2 x 0 France,2,0,0.063054
8,Brazil 2 x 2 France,2,2,0.048606
9,Brazil 1 x 3 France,1,3,0.033260


,score,home_goals,away_goals,count,simulated_probability
0,Brazil 1 x 1 France,1,1,13019,0.13019
1,Brazil 0 x 1 France,0,1,10630,0.10630
2,Brazil 1 x 0 France,1,0,10284,0.10284
3,Brazil 0 x 0 France,0,0,8598,0.08598
4,Brazil 1 x 2 France,1,2,8036,0.08036
5,Brazil 2 x 1 France,2,1,7892,0.07892
6,Brazil 0 x 2 France,0,2,6681,0.06681
7,Brazil 2 x 0 France,2,0,6314,0.06314
8,Brazil 2 x 2 France,2,2,4795,0.04795
9,Brazil 1 x 3 France,1,3,3441,0.03441
